In [ ]:
import os
import copy
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from scipy import stats
import seaborn as sns
import matplotlib.ticker as ticker
import torch
import torch.nn as nn
import random
import pickle
from collections import OrderedDict, Counter
from itertools import product
from tqdm.notebook import tqdm
import pennylane as qml
import warnings
warnings.filterwarnings("ignore")

In [ ]:
import torchani
from torchani.units import hartree2kcalmol
from torchani.nn import Sequential
from torchani.data import TransformableIterable, IterableAdapterWithLength

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

In [ ]:
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['mathtext.fontset'] = 'cm'
plt.rcParams['axes.linewidth'] = 1.0
plt.rcParams['font.size'] = 14

In [ ]:
def get_result(path):
    with open(path, 'r') as f:
        rmse_list = [float(s.strip()) for s in f.readlines()]
    return rmse_list[-100:]

# 学習曲線

## 関数

In [ ]:
def learning_plot(rmse_lists, ymin=None, ylimit=None, legend=None, title=None, name=None):
    fig = plt.figure(figsize=(7, 6))
    ax = fig.add_subplot(1, 1, 1)

    for rmse_list in rmse_lists:
        plt.plot(np.arange(1, 101), rmse_list)

    plt.xlim(0, 100)
    if ylimit is None:
        # ylimit = (max([max(l) for l in rmse_lists]) // 2 + 1) * 2
        ylimit = max([max(l) for l in rmse_lists]) + 0.2
    if ymin is None:
        ymin = 0
    plt.ylim(ymin, ylimit)
    if legend is not None:
        plt.legend(legend, fontsize=20)

    plt.tick_params(labelsize=20)
    plt.xlabel('epoch', fontsize=24)
    plt.ylabel('RMSE', fontsize=24)
    plt.xticks(np.arange(0, 101, 10))
    #plt.yscale("log")
    if title is not None:
        plt.title(title, fontsize=20)
    plt.tight_layout()
    if name is not None:
        plt.savefig(f'../results/figure/{name}.png', dpi=100)
    plt.show()

## 結果読み込み

### 古典

In [ ]:
valid_cc_s01_l1_4q = [get_result(f"../results/valid/Classic_transfer/valid_CC_s01_1layer_4qu_seed{seed}.txt") for seed in range(5)]
valid_cc_s01_l2_4q = [get_result(f"../results/valid/Classic_transfer/valid_CC_s01_2layer_4qu_seed{seed}.txt") for seed in range(5)]
valid_cc_s01_l3_4q = [get_result(f"../results/valid/Classic_transfer/valid_CC_s01_3layer_4qu_seed{seed}.txt") for seed in range(5)]
valid_cc_s01_l4_4q = [get_result(f"../results/valid/Classic_transfer/valid_CC_s01_4layer_4qu_seed{seed}.txt") for seed in range(5)]

In [ ]:
valid_cc_s01_l1_8q = [get_result(f"../results/valid/Classic_transfer/valid_CC_s01_1layer_8qu_seed{seed}.txt") for seed in range(5)]
valid_cc_s01_l2_8q = [get_result(f"../results/valid/Classic_transfer/valid_CC_s01_2layer_8qu_seed{seed}.txt") for seed in range(5)]
valid_cc_s01_l3_8q = [get_result(f"../results/valid/Classic_transfer/valid_CC_s01_3layer_8qu_seed{seed}.txt") for seed in range(5)]
valid_cc_s01_l4_8q = [get_result(f"../results/valid/Classic_transfer/valid_CC_s01_4layer_8qu_seed{seed}.txt") for seed in range(5)]

In [ ]:
valid_cc_s01_l1_12q = [get_result(f"../results/valid/Classic_transfer/valid_CC_s01_1layer_12qu_seed{seed}.txt") for seed in range(5)]
valid_cc_s01_l2_12q = [get_result(f"../results/valid/Classic_transfer/valid_CC_s01_2layer_12qu_seed{seed}.txt") for seed in range(5)]
valid_cc_s01_l3_12q = [get_result(f"../results/valid/Classic_transfer/valid_CC_s01_3layer_12qu_seed{seed}.txt") for seed in range(5)]
valid_cc_s01_l4_12q = [get_result(f"../results/valid/Classic_transfer/valid_CC_s01_4layer_12qu_seed{seed}.txt") for seed in range(5)]

In [ ]:
valid_cc_s01_l1_4q_mean = np.mean(valid_cc_s01_l1_4q, axis=0)
valid_cc_s01_l2_4q_mean = np.mean(valid_cc_s01_l2_4q, axis=0)
valid_cc_s01_l3_4q_mean = np.mean(valid_cc_s01_l3_4q, axis=0)
valid_cc_s01_l4_4q_mean = np.mean(valid_cc_s01_l4_4q, axis=0)

In [ ]:
valid_cc_s01_l1_8q_mean = np.mean(valid_cc_s01_l1_8q, axis=0)
valid_cc_s01_l2_8q_mean = np.mean(valid_cc_s01_l2_8q, axis=0)
valid_cc_s01_l3_8q_mean = np.mean(valid_cc_s01_l3_8q, axis=0)
valid_cc_s01_l4_8q_mean = np.mean(valid_cc_s01_l4_8q, axis=0)

In [ ]:
valid_cc_s01_l1_12q_mean = np.mean(valid_cc_s01_l1_12q, axis=0)
valid_cc_s01_l2_12q_mean = np.mean(valid_cc_s01_l2_12q, axis=0)
valid_cc_s01_l3_12q_mean = np.mean(valid_cc_s01_l3_12q, axis=0)
valid_cc_s01_l4_12q_mean = np.mean(valid_cc_s01_l4_12q, axis=0)

### 量子

#### Rot=RzRyRz

In [ ]:
# layer(4) x depth(2)
valid_qc_rot_s01_l1_d1_4q = [get_result(f"../results/valid/Quantum_transfer/valid_QC_s01_1layer_4qu_d1_seed{seed}_Rot.txt") for seed in range(5)]
valid_qc_rot_s01_l1_d5_4q = [get_result(f"../results/valid/Quantum_transfer/valid_QC_s01_1layer_4qu_d5_seed{seed}_Rot.txt") for seed in range(5)]
valid_qc_rot_s01_l2_d1_4q = [get_result(f"../results/valid/Quantum_transfer/valid_QC_s01_2layer_4qu_d1_seed{seed}_Rot.txt") for seed in range(5)]
valid_qc_rot_s01_l2_d5_4q = [get_result(f"../results/valid/Quantum_transfer/valid_QC_s01_2layer_4qu_d5_seed{seed}_Rot.txt") for seed in range(5)]
valid_qc_rot_s01_l3_d1_4q = [get_result(f"../results/valid/Quantum_transfer/valid_QC_s01_3layer_4qu_d1_seed{seed}_Rot.txt") for seed in range(5)]
valid_qc_rot_s01_l3_d5_4q = [get_result(f"../results/valid/Quantum_transfer/valid_QC_s01_3layer_4qu_d5_seed{seed}_Rot.txt") for seed in range(5)]
valid_qc_rot_s01_l4_d1_4q = [get_result(f"../results/valid/Quantum_transfer/valid_QC_s01_4layer_4qu_d1_seed{seed}_Rot.txt") for seed in range(5)]
valid_qc_rot_s01_l4_d5_4q = [get_result(f"../results/valid/Quantum_transfer/valid_QC_s01_4layer_4qu_d5_seed{seed}_Rot.txt") for seed in range(5)]

In [ ]:
# layer(4) x depth(2)
valid_qc_rot_s01_l1_d1_8q = [get_result(f"../results/valid/Quantum_transfer/valid_QC_s01_1layer_8qu_d1_seed{seed}_Rot.txt") for seed in range(5)]
valid_qc_rot_s01_l1_d5_8q = [get_result(f"../results/valid/Quantum_transfer/valid_QC_s01_1layer_8qu_d5_seed{seed}_Rot.txt") for seed in range(5)]
valid_qc_rot_s01_l2_d1_8q = [get_result(f"../results/valid/Quantum_transfer/valid_QC_s01_2layer_8qu_d1_seed{seed}_Rot.txt") for seed in range(5)]
valid_qc_rot_s01_l2_d5_8q = [get_result(f"../results/valid/Quantum_transfer/valid_QC_s01_2layer_8qu_d5_seed{seed}_Rot.txt") for seed in range(5)]
valid_qc_rot_s01_l3_d1_8q = [get_result(f"../results/valid/Quantum_transfer/valid_QC_s01_3layer_8qu_d1_seed{seed}_Rot.txt") for seed in range(5)]
valid_qc_rot_s01_l3_d5_8q = [get_result(f"../results/valid/Quantum_transfer/valid_QC_s01_3layer_8qu_d5_seed{seed}_Rot.txt") for seed in range(5)]
valid_qc_rot_s01_l4_d1_8q = [get_result(f"../results/valid/Quantum_transfer/valid_QC_s01_4layer_8qu_d1_seed{seed}_Rot.txt") for seed in range(5)]
valid_qc_rot_s01_l4_d5_8q = [get_result(f"../results/valid/Quantum_transfer/valid_QC_s01_4layer_8qu_d5_seed{seed}_Rot.txt") for seed in range(5)]

In [ ]:
# layer(4) x depth(2)
valid_qc_rot_s01_l1_d1_12q = [get_result(f"../results/valid/Quantum_transfer/valid_QC_s01_1layer_12qu_d1_seed{seed}_Rot.txt") for seed in range(5)]
valid_qc_rot_s01_l1_d5_12q = [get_result(f"../results/valid/Quantum_transfer/valid_QC_s01_1layer_12qu_d5_seed{seed}_Rot.txt") for seed in range(5)]
valid_qc_rot_s01_l2_d1_12q = [get_result(f"../results/valid/Quantum_transfer/valid_QC_s01_2layer_12qu_d1_seed{seed}_Rot.txt") for seed in range(5)]
valid_qc_rot_s01_l2_d5_12q = [get_result(f"../results/valid/Quantum_transfer/valid_QC_s01_2layer_12qu_d5_seed{seed}_Rot.txt") for seed in range(5)]
valid_qc_rot_s01_l3_d1_12q = [get_result(f"../results/valid/Quantum_transfer/valid_QC_s01_3layer_12qu_d1_seed{seed}_Rot.txt") for seed in range(5)]
valid_qc_rot_s01_l3_d5_12q = [get_result(f"../results/valid/Quantum_transfer/valid_QC_s01_3layer_12qu_d5_seed{seed}_Rot.txt") for seed in range(5)]
valid_qc_rot_s01_l4_d1_12q = [get_result(f"../results/valid/Quantum_transfer/valid_QC_s01_4layer_12qu_d1_seed{seed}_Rot.txt") for seed in range(5)]
valid_qc_rot_s01_l4_d5_12q = [get_result(f"../results/valid/Quantum_transfer/valid_QC_s01_4layer_12qu_d5_seed{seed}_Rot.txt") for seed in range(5)]

In [ ]:
valid_qc_rot_s01_l1_d1_4q_mean = np.mean(valid_qc_rot_s01_l1_d1_4q, axis=0)
valid_qc_rot_s01_l1_d5_4q_mean = np.mean(valid_qc_rot_s01_l1_d5_4q, axis=0)
valid_qc_rot_s01_l2_d1_4q_mean = np.mean(valid_qc_rot_s01_l2_d1_4q, axis=0)
valid_qc_rot_s01_l2_d5_4q_mean = np.mean(valid_qc_rot_s01_l2_d5_4q, axis=0)
valid_qc_rot_s01_l3_d1_4q_mean = np.mean(valid_qc_rot_s01_l3_d1_4q, axis=0)
valid_qc_rot_s01_l3_d5_4q_mean = np.mean(valid_qc_rot_s01_l3_d5_4q, axis=0)
valid_qc_rot_s01_l4_d1_4q_mean = np.mean(valid_qc_rot_s01_l4_d1_4q, axis=0)
valid_qc_rot_s01_l4_d5_4q_mean = np.mean(valid_qc_rot_s01_l4_d5_4q, axis=0)

In [ ]:
valid_qc_rot_s01_l1_d1_8q_mean = np.mean(valid_qc_rot_s01_l1_d1_8q, axis=0)
valid_qc_rot_s01_l1_d5_8q_mean = np.mean(valid_qc_rot_s01_l1_d5_8q, axis=0)
valid_qc_rot_s01_l2_d1_8q_mean = np.mean(valid_qc_rot_s01_l2_d1_8q, axis=0)
valid_qc_rot_s01_l2_d5_8q_mean = np.mean(valid_qc_rot_s01_l2_d5_8q, axis=0)
valid_qc_rot_s01_l3_d1_8q_mean = np.mean(valid_qc_rot_s01_l3_d1_8q, axis=0)
valid_qc_rot_s01_l3_d5_8q_mean = np.mean(valid_qc_rot_s01_l3_d5_8q, axis=0)
valid_qc_rot_s01_l4_d1_8q_mean = np.mean(valid_qc_rot_s01_l4_d1_8q, axis=0)
valid_qc_rot_s01_l4_d5_8q_mean = np.mean(valid_qc_rot_s01_l4_d5_8q, axis=0)

In [ ]:
valid_qc_rot_s01_l1_d1_12q_mean = np.mean(valid_qc_rot_s01_l1_d1_12q, axis=0)
valid_qc_rot_s01_l1_d5_12q_mean = np.mean(valid_qc_rot_s01_l1_d5_12q, axis=0)
valid_qc_rot_s01_l2_d1_12q_mean = np.mean(valid_qc_rot_s01_l2_d1_12q, axis=0)
valid_qc_rot_s01_l2_d5_12q_mean = np.mean(valid_qc_rot_s01_l2_d5_12q, axis=0)
valid_qc_rot_s01_l3_d1_12q_mean = np.mean(valid_qc_rot_s01_l3_d1_12q, axis=0)
valid_qc_rot_s01_l3_d5_12q_mean = np.mean(valid_qc_rot_s01_l3_d5_12q, axis=0)
valid_qc_rot_s01_l4_d1_12q_mean = np.mean(valid_qc_rot_s01_l4_d1_12q, axis=0)
valid_qc_rot_s01_l4_d5_12q_mean = np.mean(valid_qc_rot_s01_l4_d5_12q, axis=0)

#### RzRy

In [ ]:
valid_qc_zy_s01_l1_d1_4q = [get_result(f"../results/valid/Quantum_transfer/valid_QC_s01_1layer_4qu_d1_seed{seed}_RzRy.txt") for seed in range(5)]
valid_qc_zy_s01_l1_d5_4q = [get_result(f"../results/valid/Quantum_transfer/valid_QC_s01_1layer_4qu_d5_seed{seed}_RzRy.txt") for seed in range(5)]
valid_qc_zy_s01_l2_d1_4q = [get_result(f"../results/valid/Quantum_transfer/valid_QC_s01_2layer_4qu_d1_seed{seed}_RzRy.txt") for seed in range(5)]
valid_qc_zy_s01_l2_d5_4q = [get_result(f"../results/valid/Quantum_transfer/valid_QC_s01_2layer_4qu_d5_seed{seed}_RzRy.txt") for seed in range(5)]
valid_qc_zy_s01_l3_d1_4q = [get_result(f"../results/valid/Quantum_transfer/valid_QC_s01_3layer_4qu_d1_seed{seed}_RzRy.txt") for seed in range(5)]
valid_qc_zy_s01_l3_d5_4q = [get_result(f"../results/valid/Quantum_transfer/valid_QC_s01_3layer_4qu_d5_seed{seed}_RzRy.txt") for seed in range(5)]
valid_qc_zy_s01_l4_d1_4q = [get_result(f"../results/valid/Quantum_transfer/valid_QC_s01_4layer_4qu_d1_seed{seed}_RzRy.txt") for seed in range(5)]
valid_qc_zy_s01_l4_d5_4q = [get_result(f"../results/valid/Quantum_transfer/valid_QC_s01_4layer_4qu_d5_seed{seed}_RzRy.txt") for seed in range(5)]

In [ ]:
valid_qc_zy_s01_l1_d1_8q = [get_result(f"../results/valid/Quantum_transfer/valid_QC_s01_1layer_8qu_d1_seed{seed}_RzRy.txt") for seed in range(5)]
valid_qc_zy_s01_l1_d5_8q = [get_result(f"../results/valid/Quantum_transfer/valid_QC_s01_1layer_8qu_d5_seed{seed}_RzRy.txt") for seed in range(5)]
valid_qc_zy_s01_l2_d1_8q = [get_result(f"../results/valid/Quantum_transfer/valid_QC_s01_2layer_8qu_d1_seed{seed}_RzRy.txt") for seed in range(5)]
valid_qc_zy_s01_l2_d5_8q = [get_result(f"../results/valid/Quantum_transfer/valid_QC_s01_2layer_8qu_d5_seed{seed}_RzRy.txt") for seed in range(5)]
valid_qc_zy_s01_l3_d1_8q = [get_result(f"../results/valid/Quantum_transfer/valid_QC_s01_3layer_8qu_d1_seed{seed}_RzRy.txt") for seed in range(5)]
valid_qc_zy_s01_l3_d5_8q = [get_result(f"../results/valid/Quantum_transfer/valid_QC_s01_3layer_8qu_d5_seed{seed}_RzRy.txt") for seed in range(5)]
valid_qc_zy_s01_l4_d1_8q = [get_result(f"../results/valid/Quantum_transfer/valid_QC_s01_4layer_8qu_d1_seed{seed}_RzRy.txt") for seed in range(5)]
valid_qc_zy_s01_l4_d5_8q = [get_result(f"../results/valid/Quantum_transfer/valid_QC_s01_4layer_8qu_d5_seed{seed}_RzRy.txt") for seed in range(5)]

In [ ]:
valid_qc_zy_s01_l1_d1_12q = [get_result(f"../results/valid/Quantum_transfer/valid_QC_s01_1layer_12qu_d1_seed{seed}_RzRy.txt") for seed in range(5)]
valid_qc_zy_s01_l1_d5_12q = [get_result(f"../results/valid/Quantum_transfer/valid_QC_s01_1layer_12qu_d5_seed{seed}_RzRy.txt") for seed in range(5)]
valid_qc_zy_s01_l2_d1_12q = [get_result(f"../results/valid/Quantum_transfer/valid_QC_s01_2layer_12qu_d1_seed{seed}_RzRy.txt") for seed in range(5)]
valid_qc_zy_s01_l2_d5_12q = [get_result(f"../results/valid/Quantum_transfer/valid_QC_s01_2layer_12qu_d5_seed{seed}_RzRy.txt") for seed in range(5)]
valid_qc_zy_s01_l3_d1_12q = [get_result(f"../results/valid/Quantum_transfer/valid_QC_s01_3layer_12qu_d1_seed{seed}_RzRy.txt") for seed in range(5)]
valid_qc_zy_s01_l3_d5_12q = [get_result(f"../results/valid/Quantum_transfer/valid_QC_s01_3layer_12qu_d5_seed{seed}_RzRy.txt") for seed in range(5)]
valid_qc_zy_s01_l4_d1_12q = [get_result(f"../results/valid/Quantum_transfer/valid_QC_s01_4layer_12qu_d1_seed{seed}_RzRy.txt") for seed in range(5)]
valid_qc_zy_s01_l4_d5_12q = [get_result(f"../results/valid/Quantum_transfer/valid_QC_s01_4layer_12qu_d5_seed{seed}_RzRy.txt") for seed in range(5)]

In [ ]:
valid_qc_zy_s01_l1_d1_4q_mean = np.mean(valid_qc_zy_s01_l1_d1_4q, axis=0)
valid_qc_zy_s01_l1_d5_4q_mean = np.mean(valid_qc_zy_s01_l1_d5_4q, axis=0)
valid_qc_zy_s01_l2_d1_4q_mean = np.mean(valid_qc_zy_s01_l2_d1_4q, axis=0)
valid_qc_zy_s01_l2_d5_4q_mean = np.mean(valid_qc_zy_s01_l2_d5_4q, axis=0)
valid_qc_zy_s01_l3_d1_4q_mean = np.mean(valid_qc_zy_s01_l3_d1_4q, axis=0)
valid_qc_zy_s01_l3_d5_4q_mean = np.mean(valid_qc_zy_s01_l3_d5_4q, axis=0)
valid_qc_zy_s01_l4_d1_4q_mean = np.mean(valid_qc_zy_s01_l4_d1_4q, axis=0)
valid_qc_zy_s01_l4_d5_4q_mean = np.mean(valid_qc_zy_s01_l4_d5_4q, axis=0)

In [ ]:
valid_qc_zy_s01_l1_d1_8q_mean = np.mean(valid_qc_zy_s01_l1_d1_8q, axis=0)
valid_qc_zy_s01_l1_d5_8q_mean = np.mean(valid_qc_zy_s01_l1_d5_8q, axis=0)
valid_qc_zy_s01_l2_d1_8q_mean = np.mean(valid_qc_zy_s01_l2_d1_8q, axis=0)
valid_qc_zy_s01_l2_d5_8q_mean = np.mean(valid_qc_zy_s01_l2_d5_8q, axis=0)
valid_qc_zy_s01_l3_d1_8q_mean = np.mean(valid_qc_zy_s01_l3_d1_8q, axis=0)
valid_qc_zy_s01_l3_d5_8q_mean = np.mean(valid_qc_zy_s01_l3_d5_8q, axis=0)
valid_qc_zy_s01_l4_d1_8q_mean = np.mean(valid_qc_zy_s01_l4_d1_8q, axis=0)
valid_qc_zy_s01_l4_d5_8q_mean = np.mean(valid_qc_zy_s01_l4_d5_8q, axis=0)

In [ ]:
valid_qc_zy_s01_l1_d1_12q_mean = np.mean(valid_qc_zy_s01_l1_d1_12q, axis=0)
valid_qc_zy_s01_l1_d5_12q_mean = np.mean(valid_qc_zy_s01_l1_d5_12q, axis=0)
valid_qc_zy_s01_l2_d1_12q_mean = np.mean(valid_qc_zy_s01_l2_d1_12q, axis=0)
valid_qc_zy_s01_l2_d5_12q_mean = np.mean(valid_qc_zy_s01_l2_d5_12q, axis=0)
valid_qc_zy_s01_l3_d1_12q_mean = np.mean(valid_qc_zy_s01_l3_d1_12q, axis=0)
valid_qc_zy_s01_l3_d5_12q_mean = np.mean(valid_qc_zy_s01_l3_d5_12q, axis=0)
valid_qc_zy_s01_l4_d1_12q_mean = np.mean(valid_qc_zy_s01_l4_d1_12q, axis=0)
valid_qc_zy_s01_l4_d5_12q_mean = np.mean(valid_qc_zy_s01_l4_d5_12q, axis=0)

#### Ry

In [ ]:
valid_qc_y_s01_l1_d1_4q = [get_result(f"../results/valid/Quantum_transfer/valid_QC_s01_1layer_4qu_d1_seed{seed}_Ry.txt") for seed in range(5)]
valid_qc_y_s01_l1_d5_4q = [get_result(f"../results/valid/Quantum_transfer/valid_QC_s01_1layer_4qu_d5_seed{seed}_Ry.txt") for seed in range(5)]
valid_qc_y_s01_l2_d1_4q = [get_result(f"../results/valid/Quantum_transfer/valid_QC_s01_2layer_4qu_d1_seed{seed}_Ry.txt") for seed in range(5)]
valid_qc_y_s01_l2_d5_4q = [get_result(f"../results/valid/Quantum_transfer/valid_QC_s01_2layer_4qu_d5_seed{seed}_Ry.txt") for seed in range(5)]
valid_qc_y_s01_l3_d1_4q = [get_result(f"../results/valid/Quantum_transfer/valid_QC_s01_3layer_4qu_d1_seed{seed}_Ry.txt") for seed in range(5)]
valid_qc_y_s01_l3_d5_4q = [get_result(f"../results/valid/Quantum_transfer/valid_QC_s01_3layer_4qu_d5_seed{seed}_Ry.txt") for seed in range(5)]
valid_qc_y_s01_l4_d1_4q = [get_result(f"../results/valid/Quantum_transfer/valid_QC_s01_4layer_4qu_d1_seed{seed}_Ry.txt") for seed in range(5)]
valid_qc_y_s01_l4_d5_4q = [get_result(f"../results/valid/Quantum_transfer/valid_QC_s01_4layer_4qu_d5_seed{seed}_Ry.txt") for seed in range(5)]

In [ ]:
valid_qc_y_s01_l1_d1_8q = [get_result(f"../results/valid/Quantum_transfer/valid_QC_s01_1layer_8qu_d1_seed{seed}_Ry.txt") for seed in range(5)]
valid_qc_y_s01_l1_d5_8q = [get_result(f"../results/valid/Quantum_transfer/valid_QC_s01_1layer_8qu_d5_seed{seed}_Ry.txt") for seed in range(5)]
valid_qc_y_s01_l2_d1_8q = [get_result(f"../results/valid/Quantum_transfer/valid_QC_s01_2layer_8qu_d1_seed{seed}_Ry.txt") for seed in range(5)]
valid_qc_y_s01_l2_d5_8q = [get_result(f"../results/valid/Quantum_transfer/valid_QC_s01_2layer_8qu_d5_seed{seed}_Ry.txt") for seed in range(5)]
valid_qc_y_s01_l3_d1_8q = [get_result(f"../results/valid/Quantum_transfer/valid_QC_s01_3layer_8qu_d1_seed{seed}_Ry.txt") for seed in range(5)]
valid_qc_y_s01_l3_d5_8q = [get_result(f"../results/valid/Quantum_transfer/valid_QC_s01_3layer_8qu_d5_seed{seed}_Ry.txt") for seed in range(5)]
valid_qc_y_s01_l4_d1_8q = [get_result(f"../results/valid/Quantum_transfer/valid_QC_s01_4layer_8qu_d1_seed{seed}_Ry.txt") for seed in range(5)]
valid_qc_y_s01_l4_d5_8q = [get_result(f"../results/valid/Quantum_transfer/valid_QC_s01_4layer_8qu_d5_seed{seed}_Ry.txt") for seed in range(5)]

In [ ]:
valid_qc_y_s01_l1_d1_12q = [get_result(f"../results/valid/Quantum_transfer/valid_QC_s01_1layer_12qu_d1_seed{seed}_Ry.txt") for seed in range(5)]
valid_qc_y_s01_l1_d5_12q = [get_result(f"../results/valid/Quantum_transfer/valid_QC_s01_1layer_12qu_d5_seed{seed}_Ry.txt") for seed in range(5)]
valid_qc_y_s01_l2_d1_12q = [get_result(f"../results/valid/Quantum_transfer/valid_QC_s01_2layer_12qu_d1_seed{seed}_Ry.txt") for seed in range(5)]
valid_qc_y_s01_l2_d5_12q = [get_result(f"../results/valid/Quantum_transfer/valid_QC_s01_2layer_12qu_d5_seed{seed}_Ry.txt") for seed in range(5)]
valid_qc_y_s01_l3_d1_12q = [get_result(f"../results/valid/Quantum_transfer/valid_QC_s01_3layer_12qu_d1_seed{seed}_Ry.txt") for seed in range(5)]
valid_qc_y_s01_l3_d5_12q = [get_result(f"../results/valid/Quantum_transfer/valid_QC_s01_3layer_12qu_d5_seed{seed}_Ry.txt") for seed in range(5)]
valid_qc_y_s01_l4_d1_12q = [get_result(f"../results/valid/Quantum_transfer/valid_QC_s01_4layer_12qu_d1_seed{seed}_Ry.txt") for seed in range(5)]
valid_qc_y_s01_l4_d5_12q = [get_result(f"../results/valid/Quantum_transfer/valid_QC_s01_4layer_12qu_d5_seed{seed}_Ry.txt") for seed in range(5)]

In [ ]:
valid_qc_y_s01_l1_d1_4q_mean = np.mean(valid_qc_y_s01_l1_d1_4q, axis=0)
valid_qc_y_s01_l1_d5_4q_mean = np.mean(valid_qc_y_s01_l1_d5_4q, axis=0)
valid_qc_y_s01_l2_d1_4q_mean = np.mean(valid_qc_y_s01_l2_d1_4q, axis=0)
valid_qc_y_s01_l2_d5_4q_mean = np.mean(valid_qc_y_s01_l2_d5_4q, axis=0)
valid_qc_y_s01_l3_d1_4q_mean = np.mean(valid_qc_y_s01_l3_d1_4q, axis=0)
valid_qc_y_s01_l3_d5_4q_mean = np.mean(valid_qc_y_s01_l3_d5_4q, axis=0)
valid_qc_y_s01_l4_d1_4q_mean = np.mean(valid_qc_y_s01_l4_d1_4q, axis=0)
valid_qc_y_s01_l4_d5_4q_mean = np.mean(valid_qc_y_s01_l4_d5_4q, axis=0)

In [ ]:
valid_qc_y_s01_l1_d1_8q_mean = np.mean(valid_qc_y_s01_l1_d1_8q, axis=0)
valid_qc_y_s01_l1_d5_8q_mean = np.mean(valid_qc_y_s01_l1_d5_8q, axis=0)
valid_qc_y_s01_l2_d1_8q_mean = np.mean(valid_qc_y_s01_l2_d1_8q, axis=0)
valid_qc_y_s01_l2_d5_8q_mean = np.mean(valid_qc_y_s01_l2_d5_8q, axis=0)
valid_qc_y_s01_l3_d1_8q_mean = np.mean(valid_qc_y_s01_l3_d1_8q, axis=0)
valid_qc_y_s01_l3_d5_8q_mean = np.mean(valid_qc_y_s01_l3_d5_8q, axis=0)
valid_qc_y_s01_l4_d1_8q_mean = np.mean(valid_qc_y_s01_l4_d1_8q, axis=0)
valid_qc_y_s01_l4_d5_8q_mean = np.mean(valid_qc_y_s01_l4_d5_8q, axis=0)

In [ ]:
valid_qc_y_s01_l1_d1_12q_mean = np.mean(valid_qc_y_s01_l1_d1_12q, axis=0)
valid_qc_y_s01_l1_d5_12q_mean = np.mean(valid_qc_y_s01_l1_d5_12q, axis=0)
valid_qc_y_s01_l2_d1_12q_mean = np.mean(valid_qc_y_s01_l2_d1_12q, axis=0)
valid_qc_y_s01_l2_d5_12q_mean = np.mean(valid_qc_y_s01_l2_d5_12q, axis=0)
valid_qc_y_s01_l3_d1_12q_mean = np.mean(valid_qc_y_s01_l3_d1_12q, axis=0)
valid_qc_y_s01_l3_d5_12q_mean = np.mean(valid_qc_y_s01_l3_d5_12q, axis=0)
valid_qc_y_s01_l4_d1_12q_mean = np.mean(valid_qc_y_s01_l4_d1_12q, axis=0)
valid_qc_y_s01_l4_d5_12q_mean = np.mean(valid_qc_y_s01_l4_d5_12q, axis=0)

## プロット

In [ ]:
cc_valid_rmse_dict = {
    "s01_l1_4q": valid_cc_s01_l1_4q_mean,
    "s01_l2_4q": valid_cc_s01_l2_4q_mean,
    "s01_l3_4q": valid_cc_s01_l3_4q_mean,
    "s01_l4_4q": valid_cc_s01_l4_4q_mean,
    "s01_l1_8q": valid_cc_s01_l1_8q_mean,
    "s01_l2_8q": valid_cc_s01_l2_8q_mean,
    "s01_l3_8q": valid_cc_s01_l3_8q_mean,
    "s01_l4_8q": valid_cc_s01_l4_8q_mean,
    "s01_l1_12q": valid_cc_s01_l1_12q_mean,
    "s01_l2_12q": valid_cc_s01_l2_12q_mean,
    "s01_l3_12q": valid_cc_s01_l3_12q_mean,
    "s01_l4_12q": valid_cc_s01_l4_12q_mean,
}
ry_valid_rmse_dict = {
    "s01_l1_d1_4q": valid_qc_y_s01_l1_d1_4q_mean,
    "s01_l2_d1_4q": valid_qc_y_s01_l2_d1_4q_mean,
    "s01_l3_d1_4q": valid_qc_y_s01_l3_d1_4q_mean,
    "s01_l4_d1_4q": valid_qc_y_s01_l4_d1_4q_mean,
    "s01_l1_d1_8q": valid_qc_y_s01_l1_d1_8q_mean,
    "s01_l2_d1_8q": valid_qc_y_s01_l2_d1_8q_mean,
    "s01_l3_d1_8q": valid_qc_y_s01_l3_d1_8q_mean,
    "s01_l4_d1_8q": valid_qc_y_s01_l4_d1_8q_mean,
    "s01_l1_d1_12q": valid_qc_y_s01_l1_d1_12q_mean,
    "s01_l2_d1_12q": valid_qc_y_s01_l2_d1_12q_mean,
    "s01_l3_d1_12q": valid_qc_y_s01_l3_d1_12q_mean,
    "s01_l4_d1_12q": valid_qc_y_s01_l4_d1_12q_mean,
    "s01_l1_d5_4q": valid_qc_y_s01_l1_d5_4q_mean,
    "s01_l2_d5_4q": valid_qc_y_s01_l2_d5_4q_mean,
    "s01_l3_d5_4q": valid_qc_y_s01_l3_d5_4q_mean,
    "s01_l4_d5_4q": valid_qc_y_s01_l4_d5_4q_mean,
    "s01_l1_d5_8q": valid_qc_y_s01_l1_d5_8q_mean,
    "s01_l2_d5_8q": valid_qc_y_s01_l2_d5_8q_mean,
    "s01_l3_d5_8q": valid_qc_y_s01_l3_d5_8q_mean,
    "s01_l4_d5_8q": valid_qc_y_s01_l4_d5_8q_mean,
    "s01_l1_d5_12q": valid_qc_y_s01_l1_d5_12q_mean,
    "s01_l2_d5_12q": valid_qc_y_s01_l2_d5_12q_mean,
    "s01_l3_d5_12q": valid_qc_y_s01_l3_d5_12q_mean,
    "s01_l4_d5_12q": valid_qc_y_s01_l4_d5_12q_mean,
}
rzry_valid_rmse_dict = {
    "s01_l1_d1_4q": valid_qc_zy_s01_l1_d1_4q_mean,
    "s01_l2_d1_4q": valid_qc_zy_s01_l2_d1_4q_mean,
    "s01_l3_d1_4q": valid_qc_zy_s01_l3_d1_4q_mean,
    "s01_l4_d1_4q": valid_qc_zy_s01_l4_d1_4q_mean,
    "s01_l1_d1_8q": valid_qc_zy_s01_l1_d1_8q_mean,
    "s01_l2_d1_8q": valid_qc_zy_s01_l2_d1_8q_mean,
    "s01_l3_d1_8q": valid_qc_zy_s01_l3_d1_8q_mean,
    "s01_l4_d1_8q": valid_qc_zy_s01_l4_d1_8q_mean,
    "s01_l1_d1_12q": valid_qc_zy_s01_l1_d1_12q_mean,
    "s01_l2_d1_12q": valid_qc_zy_s01_l2_d1_12q_mean,
    "s01_l3_d1_12q": valid_qc_zy_s01_l3_d1_12q_mean,
    "s01_l4_d1_12q": valid_qc_zy_s01_l4_d1_12q_mean,
    "s01_l1_d5_4q": valid_qc_zy_s01_l1_d5_4q_mean,
    "s01_l2_d5_4q": valid_qc_zy_s01_l2_d5_4q_mean,
    "s01_l3_d5_4q": valid_qc_zy_s01_l3_d5_4q_mean,
    "s01_l4_d5_4q": valid_qc_zy_s01_l4_d5_4q_mean,
    "s01_l1_d5_8q": valid_qc_zy_s01_l1_d5_8q_mean,
    "s01_l2_d5_8q": valid_qc_zy_s01_l2_d5_8q_mean,
    "s01_l3_d5_8q": valid_qc_zy_s01_l3_d5_8q_mean,
    "s01_l4_d5_8q": valid_qc_zy_s01_l4_d5_8q_mean,
    "s01_l1_d5_12q": valid_qc_zy_s01_l1_d5_12q_mean,
    "s01_l2_d5_12q": valid_qc_zy_s01_l2_d5_12q_mean,
    "s01_l3_d5_12q": valid_qc_zy_s01_l3_d5_12q_mean,
    "s01_l4_d5_12q": valid_qc_zy_s01_l4_d5_12q_mean,
}

In [ ]:
# 4qu
cc_valid_rmse_dict = {
    "s01_l1_4q": valid_cc_s01_l1_4q_mean,
    "s01_l2_4q": valid_cc_s01_l2_4q_mean,
    "s01_l3_4q": valid_cc_s01_l3_4q_mean,
    "s01_l4_4q": valid_cc_s01_l4_4q_mean,
}
ry_valid_rmse_dict = {
    "s01_l1_d1_4q": valid_qc_y_s01_l1_d1_4q_mean,
    "s01_l2_d1_4q": valid_qc_y_s01_l2_d1_4q_mean,
    "s01_l3_d1_4q": valid_qc_y_s01_l3_d1_4q_mean,
    "s01_l4_d1_4q": valid_qc_y_s01_l4_d1_4q_mean,
    "s01_l1_d5_4q": valid_qc_y_s01_l1_d5_4q_mean,
    "s01_l2_d5_4q": valid_qc_y_s01_l2_d5_4q_mean,
    "s01_l3_d5_4q": valid_qc_y_s01_l3_d5_4q_mean,
    "s01_l4_d5_4q": valid_qc_y_s01_l4_d5_4q_mean,
}
rzry_valid_rmse_dict = {
    "s01_l1_d1_4q": valid_qc_zy_s01_l1_d1_4q_mean,
    "s01_l2_d1_4q": valid_qc_zy_s01_l2_d1_4q_mean,
    "s01_l3_d1_4q": valid_qc_zy_s01_l3_d1_4q_mean,
    "s01_l4_d1_4q": valid_qc_zy_s01_l4_d1_4q_mean,
    "s01_l1_d5_4q": valid_qc_zy_s01_l1_d5_4q_mean,
    "s01_l2_d5_4q": valid_qc_zy_s01_l2_d5_4q_mean,
    "s01_l3_d5_4q": valid_qc_zy_s01_l3_d5_4q_mean,
    "s01_l4_d5_4q": valid_qc_zy_s01_l4_d5_4q_mean,
}

In [ ]:
ylimit_dict = {"1layer": 4.0, "2layer": 1.3, "3layer": 1.3, "4layer": 1.3}
ymin_dict = {"1layer": 1.0, "2layer": 0.4, "3layer": 0.4, "4layer": 0.4}

In [ ]:
def get_learning_plot(data_No, n_layer, n_qubit, q_depth):
    ylimit = ylimit_dict[f"{n_layer}layer"]
    ymin = ymin_dict[f"{n_layer}layer"]

    learning_plot([
            cc_valid_rmse_dict[f"s0{data_No}_l{n_layer}_{n_qubit}q"],
            ry_valid_rmse_dict[f"s0{data_No}_l{n_layer}_d{q_depth}_{n_qubit}q"],
            rzry_valid_rmse_dict[f"s0{data_No}_l{n_layer}_d{q_depth}_{n_qubit}q"],
        ],
        legend=["Classical", "Ry", "RzRy"],
        title=f"Data: {data_No} / Layer: {n_layer} / Qubit: {n_qubit} / Q_depth: {q_depth}",
        ylimit=ylimit,
        ymin=ymin,
        #name=f"learning_plot_s0{data_No}_l{n_layer}_d{q_depth}_{n_qubit}q"
        name=f"learning_curve_l{n_layer}_d{q_depth}"
    )

In [ ]:
sns.set(style="darkgrid")

### s01, 4qubit

In [ ]:
for n_layer in range(1, 5):
    #for n_qubit in [4, 8, 12]:
    n_qubit = 4
    for q_depth in [1, 5]:
        get_learning_plot(1, n_layer, n_qubit, q_depth)

## Rotあり

In [ ]:
data_No = 1
n_qubit = 4

In [ ]:
n_layer = 1
q_depth = 1
ylimit = ylimit_dict[f"{n_layer}layer"]
ymin = ymin_dict[f"{n_layer}layer"]

learning_plot(
    [
        valid_cc_s01_l1_4q_mean,
        valid_qc_y_s01_l1_d1_4q_mean,
        valid_qc_zy_s01_l1_d1_4q_mean,
        valid_qc_rot_s01_l1_d1_4q_mean
    ],
    legend=["Classic", "Ry", "RzRy", "RzRyRz"],
    title=f"Data {data_No} / Layer: {n_layer} / Qubit: {n_qubit} / Q_depth: {q_depth}",
    ylimit=ylimit, ymin=ymin,
    name=f"learning_plot_s0{data_No}_l{n_layer}_d{q_depth}_{n_qubit}q"
)

q_depth = 5
learning_plot(
    [
        valid_cc_s01_l1_4q_mean,
        valid_qc_y_s01_l1_d5_4q_mean,
        valid_qc_zy_s01_l1_d5_4q_mean,
        valid_qc_rot_s01_l1_d5_4q_mean
    ],
    legend=["Classic", "Ry", "RzRy", "RzRyRz"],
    title=f"Data {data_No} / Layer: {n_layer} / Qubit: {n_qubit} / Q_depth: {q_depth}",
    ylimit=ylimit, ymin=ymin,
    name=f"learning_plot_s0{data_No}_l{n_layer}_d{q_depth}_{n_qubit}q"
)

In [ ]:
n_layer = 2
q_depth = 1
ylimit = ylimit_dict[f"{n_layer}layer"]
ymin = ymin_dict[f"{n_layer}layer"]

learning_plot(
    [
        valid_cc_s01_l2_4q_mean,
        valid_qc_y_s01_l2_d1_4q_mean,
        valid_qc_zy_s01_l2_d1_4q_mean,
        valid_qc_rot_s01_l2_d1_4q_mean
    ],
    legend=["Classic", "Ry", "RzRy", "RzRyRz"],
    title=f"Data {data_No} / Layer: {n_layer} / Qubit: {n_qubit} / Q_depth: {q_depth}",
    ylimit=ylimit, ymin=ymin,
    name=f"learning_plot_s0{data_No}_l{n_layer}_d{q_depth}_{n_qubit}q"
)

q_depth = 5
learning_plot(
    [
        valid_cc_s01_l2_4q_mean,
        valid_qc_y_s01_l2_d5_4q_mean,
        valid_qc_zy_s01_l2_d5_4q_mean,
        valid_qc_rot_s01_l2_d5_4q_mean
    ],
    legend=["Classic", "Ry", "RzRy", "RzRyRz"],
    title=f"Data {data_No} / Layer: {n_layer} / Qubit: {n_qubit} / Q_depth: {q_depth}",
    ylimit=ylimit, ymin=ymin,
    name=f"learning_plot_s0{data_No}_l{n_layer}_d{q_depth}_{n_qubit}q"
)

In [ ]:
n_layer = 3
q_depth = 1
ylimit = ylimit_dict[f"{n_layer}layer"]
ymin = ymin_dict[f"{n_layer}layer"]

learning_plot(
    [
        valid_cc_s01_l3_4q_mean,
        valid_qc_y_s01_l3_d1_4q_mean,
        valid_qc_zy_s01_l3_d1_4q_mean,
        valid_qc_rot_s01_l3_d1_4q_mean
    ],
    legend=["Classic", "Ry", "RzRy", "RzRyRz"],
    title=f"Data {data_No} / Layer: {n_layer} / Qubit: {n_qubit} / Q_depth: {q_depth}",
    ylimit=ylimit, ymin=ymin,
    name=f"learning_plot_s0{data_No}_l{n_layer}_d{q_depth}_{n_qubit}q"
)

q_depth = 5
learning_plot(
    [
        valid_cc_s01_l3_4q_mean,
        valid_qc_y_s01_l3_d5_4q_mean,
        valid_qc_zy_s01_l3_d5_4q_mean,
        valid_qc_rot_s01_l3_d5_4q_mean
    ],
    legend=["Classic", "Ry", "RzRy", "RzRyRz"],
    title=f"Data {data_No} / Layer: {n_layer} / Qubit: {n_qubit} / Q_depth: {q_depth}",
    ylimit=ylimit, ymin=ymin,
    name=f"learning_plot_s0{data_No}_l{n_layer}_d{q_depth}_{n_qubit}q"
)

In [ ]:
n_layer = 4
q_depth = 1
ylimit = ylimit_dict[f"{n_layer}layer"]
ymin = ymin_dict[f"{n_layer}layer"]

learning_plot(
    [
        valid_cc_s01_l4_4q_mean,
        valid_qc_y_s01_l4_d1_4q_mean,
        valid_qc_zy_s01_l4_d1_4q_mean,
        valid_qc_rot_s01_l4_d1_4q_mean
    ],
    legend=["Classic", "Ry", "RzRy", "RzRyRz"],
    title=f"Data {data_No} / Layer: {n_layer} / Qubit: {n_qubit} / Q_depth: {q_depth}",
    ylimit=ylimit, ymin=ymin,
    name=f"learning_plot_s0{data_No}_l{n_layer}_d{q_depth}_{n_qubit}q"
)

q_depth = 5
learning_plot(
    [
        valid_cc_s01_l4_4q_mean,
        valid_qc_y_s01_l4_d5_4q_mean,
        valid_qc_zy_s01_l4_d5_4q_mean,
        valid_qc_rot_s01_l4_d5_4q_mean
    ],
    legend=["Classic", "Ry", "RzRy", "RzRyRz"],
    title=f"Data {data_No} / Layer: {n_layer} / Qubit: {n_qubit} / Q_depth: {q_depth}",
    ylimit=ylimit, ymin=ymin,
    name=f"learning_plot_s0{data_No}_l{n_layer}_d{q_depth}_{n_qubit}q"
)

### s01, 8qubit

In [ ]:
data_No = 1
n_qubit = 8

In [ ]:
n_layer = 1
q_depth = 1
ylimit = ylimit_dict[f"{n_layer}layer"]
ymin = ymin_dict[f"{n_layer}layer"]

learning_plot(
    [
        valid_cc_s01_l1_8q_mean,
        valid_qc_y_s01_l1_d1_8q_mean,
        valid_qc_zy_s01_l1_d1_8q_mean,
        valid_qc_rot_s01_l1_d1_8q_mean
    ],
    legend=["Classic", "Ry", "RzRy", "RzRyRz"],
    title=f"Data {data_No} / Layer: {n_layer} / Qubit: {n_qubit} / Q_depth: {q_depth}",
    ylimit=ylimit, ymin=ymin,
    name=f"learning_plot_s0{data_No}_l{n_layer}_d{q_depth}_{n_qubit}q"
)

q_depth = 5
learning_plot(
    [
        valid_cc_s01_l1_8q_mean,
        valid_qc_y_s01_l1_d5_8q_mean,
        valid_qc_zy_s01_l1_d5_8q_mean,
        valid_qc_rot_s01_l1_d5_8q_mean
    ],
    legend=["Classic", "Ry", "RzRy", "RzRyRz"],
    title=f"Data {data_No} / Layer: {n_layer} / Qubit: {n_qubit} / Q_depth: {q_depth}",
    ylimit=ylimit, ymin=ymin,
    name=f"learning_plot_s0{data_No}_l{n_layer}_d{q_depth}_{n_qubit}q"
)

In [ ]:
n_layer = 2
q_depth = 1
ylimit = ylimit_dict[f"{n_layer}layer"]
ymin = ymin_dict[f"{n_layer}layer"]

learning_plot(
    [
        valid_cc_s01_l2_8q_mean,
        valid_qc_y_s01_l2_d1_8q_mean,
        valid_qc_zy_s01_l2_d1_8q_mean,
        valid_qc_rot_s01_l2_d1_8q_mean
    ],
    legend=["Classic", "Ry", "RzRy", "RzRyRz"],
    title=f"Data {data_No} / Layer: {n_layer} / Qubit: {n_qubit} / Q_depth: {q_depth}",
    ylimit=ylimit, ymin=ymin,
    name=f"learning_plot_s0{data_No}_l{n_layer}_d{q_depth}_{n_qubit}q"
)

q_depth = 5
learning_plot(
    [
        valid_cc_s01_l2_8q_mean,
        valid_qc_y_s01_l2_d5_8q_mean,
        valid_qc_zy_s01_l2_d5_8q_mean,
        valid_qc_rot_s01_l2_d5_8q_mean
    ],
    legend=["Classic", "Ry", "RzRy", "RzRyRz"],
    title=f"Data {data_No} / Layer: {n_layer} / Qubit: {n_qubit} / Q_depth: {q_depth}",
    ylimit=ylimit, ymin=ymin,
    name=f"learning_plot_s0{data_No}_l{n_layer}_d{q_depth}_{n_qubit}q"
)

In [ ]:
n_layer = 3
q_depth = 1
ylimit = ylimit_dict[f"{n_layer}layer"]
ymin = ymin_dict[f"{n_layer}layer"]

learning_plot(
    [
        valid_cc_s01_l3_8q_mean,
        valid_qc_y_s01_l3_d1_8q_mean,
        valid_qc_zy_s01_l3_d1_8q_mean,
        valid_qc_rot_s01_l3_d1_8q_mean
    ],
    legend=["Classic", "Ry", "RzRy", "RzRyRz"],
    title=f"Data {data_No} / Layer: {n_layer} / Qubit: {n_qubit} / Q_depth: {q_depth}",
    ylimit=ylimit, ymin=ymin,
    name=f"learning_plot_s0{data_No}_l{n_layer}_d{q_depth}_{n_qubit}q"
)

q_depth = 5
learning_plot(
    [
        valid_cc_s01_l3_8q_mean,
        valid_qc_y_s01_l3_d5_8q_mean,
        valid_qc_zy_s01_l3_d5_8q_mean,
        valid_qc_rot_s01_l3_d5_8q_mean
    ],
    legend=["Classic", "Ry", "RzRy", "RzRyRz"],
    title=f"Data {data_No} / Layer: {n_layer} / Qubit: {n_qubit} / Q_depth: {q_depth}",
    ylimit=ylimit, ymin=ymin,
    name=f"learning_plot_s0{data_No}_l{n_layer}_d{q_depth}_{n_qubit}q"
)

In [ ]:
n_layer = 4
q_depth = 1
ylimit = ylimit_dict[f"{n_layer}layer"]
ymin = ymin_dict[f"{n_layer}layer"]

learning_plot(
    [
        valid_cc_s01_l4_8q_mean,
        valid_qc_y_s01_l4_d1_8q_mean,
        valid_qc_zy_s01_l4_d1_8q_mean,
        valid_qc_rot_s01_l4_d1_8q_mean
    ],
    legend=["Classic", "Ry", "RzRy", "RzRyRz"],
    title=f"Data {data_No} / Layer: {n_layer} / Qubit: {n_qubit} / Q_depth: {q_depth}",
    ylimit=ylimit, ymin=ymin,
    name=f"learning_plot_s0{data_No}_l{n_layer}_d{q_depth}_{n_qubit}q"
)

q_depth = 5
learning_plot(
    [
        valid_cc_s01_l4_8q_mean,
        valid_qc_y_s01_l4_d5_8q_mean,
        valid_qc_zy_s01_l4_d5_8q_mean,
        valid_qc_rot_s01_l4_d5_8q_mean
    ],
    legend=["Classic", "Ry", "RzRy", "RzRyRz"],
    title=f"Data {data_No} / Layer: {n_layer} / Qubit: {n_qubit} / Q_depth: {q_depth}",
    ylimit=ylimit, ymin=ymin,
    name=f"learning_plot_s0{data_No}_l{n_layer}_d{q_depth}_{n_qubit}q"
)

### s01, 12qubit

In [ ]:
data_No = 1
n_qubit = 12

In [ ]:
n_layer = 1
q_depth = 1
ylimit = ylimit_dict[f"{n_layer}layer"]
ymin = ymin_dict[f"{n_layer}layer"]

learning_plot(
    [
        valid_cc_s01_l1_12q_mean,
        valid_qc_y_s01_l1_d1_12q_mean,
        valid_qc_zy_s01_l1_d1_12q_mean,
        valid_qc_rot_s01_l1_d1_12q_mean
    ],
    legend=["Classic", "Ry", "RzRy", "RzRyRz"],
    title=f"Data {data_No} / Layer: {n_layer} / Qubit: {n_qubit} / Q_depth: {q_depth}",
    ylimit=ylimit, ymin=ymin,
    name=f"learning_plot_s0{data_No}_l{n_layer}_d{q_depth}_{n_qubit}q"
)

q_depth = 5
learning_plot(
    [
        valid_cc_s01_l1_12q_mean,
        valid_qc_y_s01_l1_d5_12q_mean,
        valid_qc_zy_s01_l1_d5_12q_mean,
        valid_qc_rot_s01_l1_d5_12q_mean
    ],
    legend=["Classic", "Ry", "RzRy", "RzRyRz"],
    title=f"Data {data_No} / Layer: {n_layer} / Qubit: {n_qubit} / Q_depth: {q_depth}",
    ylimit=ylimit, ymin=ymin,
    name=f"learning_plot_s0{data_No}_l{n_layer}_d{q_depth}_{n_qubit}q"
)

In [ ]:
n_layer = 2
q_depth = 1
ylimit = ylimit_dict[f"{n_layer}layer"]
ymin = ymin_dict[f"{n_layer}layer"]

learning_plot(
    [
        valid_cc_s01_l2_12q_mean,
        valid_qc_y_s01_l2_d1_12q_mean,
        valid_qc_zy_s01_l2_d1_12q_mean,
        valid_qc_rot_s01_l2_d1_12q_mean
    ],
    legend=["Classic", "Ry", "RzRy", "RzRyRz"],
    title=f"Data {data_No} / Layer: {n_layer} / Qubit: {n_qubit} / Q_depth: {q_depth}",
    ylimit=ylimit, ymin=ymin,
    name=f"learning_plot_s0{data_No}_l{n_layer}_d{q_depth}_{n_qubit}q"
)

q_depth = 5
learning_plot(
    [
        valid_cc_s01_l2_12q_mean,
        valid_qc_y_s01_l2_d5_12q_mean,
        valid_qc_zy_s01_l2_d5_12q_mean,
        valid_qc_rot_s01_l2_d5_12q_mean
    ],
    legend=["Classic", "Ry", "RzRy", "RzRyRz"],
    title=f"Data {data_No} / Layer: {n_layer} / Qubit: {n_qubit} / Q_depth: {q_depth}",
    ylimit=ylimit, ymin=ymin,
    name=f"learning_plot_s0{data_No}_l{n_layer}_d{q_depth}_{n_qubit}q"
)

In [ ]:
n_layer = 3
q_depth = 1
ylimit = ylimit_dict[f"{n_layer}layer"]
ymin = ymin_dict[f"{n_layer}layer"]

learning_plot(
    [
        valid_cc_s01_l3_12q_mean,
        valid_qc_y_s01_l3_d1_12q_mean,
        valid_qc_zy_s01_l3_d1_12q_mean,
        valid_qc_rot_s01_l3_d1_12q_mean
    ],
    legend=["Classic", "Ry", "RzRy", "RzRyRz"],
    title=f"Data {data_No} / Layer: {n_layer} / Qubit: {n_qubit} / Q_depth: {q_depth}",
    ylimit=ylimit, ymin=ymin,
    name=f"learning_plot_s0{data_No}_l{n_layer}_d{q_depth}_{n_qubit}q"
)

q_depth = 5
learning_plot(
    [
        valid_cc_s01_l3_12q_mean,
        valid_qc_y_s01_l3_d5_12q_mean,
        valid_qc_zy_s01_l3_d5_12q_mean,
        valid_qc_rot_s01_l3_d5_12q_mean
    ],
    legend=["Classic", "Ry", "RzRy", "RzRyRz"],
    title=f"Data {data_No} / Layer: {n_layer} / Qubit: {n_qubit} / Q_depth: {q_depth}",
    ylimit=ylimit, ymin=ymin,
    name=f"learning_plot_s0{data_No}_l{n_layer}_d{q_depth}_{n_qubit}q"
)

In [ ]:
n_layer = 4
q_depth = 1
ylimit = ylimit_dict[f"{n_layer}layer"]
ymin = ymin_dict[f"{n_layer}layer"]

learning_plot(
    [
        valid_cc_s01_l4_12q_mean,
        valid_qc_y_s01_l4_d1_12q_mean,
        valid_qc_zy_s01_l4_d1_12q_mean,
        valid_qc_rot_s01_l4_d1_12q_mean
    ],
    legend=["Classic", "Ry", "RzRy", "RzRyRz"],
    title=f"Data {data_No} / Layer: {n_layer} / Qubit: {n_qubit} / Q_depth: {q_depth}",
    ylimit=ylimit, ymin=ymin,
    name=f"learning_plot_s0{data_No}_l{n_layer}_d{q_depth}_{n_qubit}q"
)

q_depth = 5
learning_plot(
    [
        valid_cc_s01_l4_12q_mean,
        valid_qc_y_s01_l4_d5_12q_mean,
        valid_qc_zy_s01_l4_d5_12q_mean,
        valid_qc_rot_s01_l4_d5_12q_mean
    ],
    legend=["Classic", "Ry", "RzRy", "RzRyRz"],
    title=f"Data {data_No} / Layer: {n_layer} / Qubit: {n_qubit} / Q_depth: {q_depth}",
    ylimit=ylimit, ymin=ymin,
    name=f"learning_plot_s0{data_No}_l{n_layer}_d{q_depth}_{n_qubit}q"
)

# 改善度を比較

In [ ]:
baseline_dict = {}
for data_No in range(1, 5):
    for nqubit in [4, 8, 12]:
        for n_layer in range(1, 5):
            path_list = [f"../results/valid/Classic_transfer/valid_CC_s0{data_No}_{n_layer}layer_{nqubit}qu_seed{seed}.txt" for seed in range(5)]
            cc_rmse_start = [get_result(path)[0] for path in path_list]
            start_mean = np.mean(cc_rmse_start)
            baseline_dict[f"s{data_No}_{nqubit}q_{n_layer}layer"] = start_mean

In [ ]:
baseline_dict

In [ ]:
cc_rmse

In [ ]:
qc_rmse_RzRy

In [ ]:
with open("../results/test/CC_rmse_dict.pickle", "rb") as f:
    cc_rmse = pickle.load(f)
with open("../results/test/QC_rmse_dict_Ry.pickle", "rb") as f:
    qc_rmse_Ry = pickle.load(f)
with open("../results/test/QC_rmse_dict_RzRy.pickle", "rb") as f:
    qc_rmse_RzRy = pickle.load(f)
with open("../results/test/QC_rmse_dict_Rot.pickle", "rb") as f:
    qc_rmse_Rot = pickle.load(f)

In [ ]:
import openpyxl

In [ ]:
rmse_sheet = pd.read_excel("../rmse_raw.xlsx", sheet_name=[f"dataset{i}" for i in range(1, 5)], index_col=[0, 1], header=[0, 1, 2])
for sheet_No in range(1, 5):
    for i, n in enumerate([4, 8, 12]):
        for l in range(1, 5):
            rmse_sheet[f"dataset{sheet_No}"].iloc[i * 20 + (l - 1) * 5:i * 20 + l * 5, 0] = cc_rmse[f"s0{sheet_No}_{n}qu_{l}layer"]
            for j, d in enumerate([1, 5]):
                rmse_sheet[f"dataset{sheet_No}"].iloc[i * 20 + (l - 1) * 5:i * 20 + l * 5, 1 + j] = qc_rmse_Ry[f"s0{sheet_No}_{n}qu_{l}layer_{d}depth"]
                rmse_sheet[f"dataset{sheet_No}"].iloc[i * 20 + (l - 1) * 5:i * 20 + l * 5, 3 + j] = qc_rmse_RzRy[f"s0{sheet_No}_{n}qu_{l}layer_{d}depth"]

In [ ]:
with pd.ExcelWriter("../rmse_raw_data.xlsx") as w:
    for sheet_No in range(1, 5):
        rmse_sheet[f"dataset{sheet_No}"].to_excel(w, sheet_name=f"dataset{sheet_No}")

In [ ]:
rmse_sheet

In [ ]:
cc_rmse

def calc_improve(baseline, rmse_list):
    improve_list = (baseline - np.array(rmse_list)) / baseline * 100
    return improve_list

In [ ]:
def calc_improve_value(rmse_dict, data_No, n_qubit, n_layer, q_depth=None):
    key = f"s0{data_No}_{n_qubit}qu_{n_layer}layer"
    if q_depth is not None:
        key += f"_{q_depth}depth"
    rmse_list = rmse_dict[key]
    improve_list = baseline_dict[f"s{data_No}_{n_qubit}q_{n_layer}layer"] - np.array(rmse_list)
    return improve_list

In [ ]:
baseline

In [ ]:
data_No = 1

In [ ]:
def plot_rmse(data_No, n_qubit, q_depth, ymax=None, ymin=None, yrange=None):
    df_rmse = pd.DataFrame(columns=["RMSE (kcal/mol)", "n_layer", "Parameter"])

    base_key = f"s0{data_No}_{n_qubit}qu"

    for n_layer in range(1, 5):
        cc_key = base_key + f"_{n_layer}layer"
        qc_key = cc_key + f"_{q_depth}depth"

        df_rmse = pd.concat([df_rmse, pd.DataFrame({"RMSE (kcal/mol)": cc_rmse[cc_key], "n_layer": n_layer, "Parameter": "Classical"})])
        df_rmse = pd.concat([df_rmse, pd.DataFrame({"RMSE (kcal/mol)": qc_rmse_Ry[qc_key], "n_layer": n_layer, "Parameter": "Ry"})])
        df_rmse = pd.concat([df_rmse, pd.DataFrame({"RMSE (kcal/mol)": qc_rmse_RzRy[qc_key], "n_layer": n_layer, "Parameter": "RzRy"})])
        #df_rmse = pd.concat([df_rmse, pd.DataFrame({"RMSE (kcal/mol)": qc_rmse_Rot[qc_key], "n_layer": n_layer, "Parameter": "RzRyRz"})])

    fig, ax1 = plt.subplots()
    sns.set(style="darkgrid")
    sns.barplot(x="n_layer", y="RMSE (kcal/mol)", hue="Parameter", data=df_rmse)
    plt.axhline(1.0, color="k", ls="--")

    if ymin is None:
        ymin = 0.3
    if ymax is None:
        ymax = max(df_rmse["RMSE (kcal/mol)"].values) * 1.3
    plt.ylim(ymin, ymax)
    plt.xlabel("# of layer" + " " + r"$(l)$")
    #plt.yscale("log")
    #tick_list = list(np.arange(math.floor(ymax))).insert(1, 0.5)
    if yrange is not None:
        plt.yticks(yrange)
    plt.gca().yaxis.set_minor_formatter(ticker.NullFormatter())
    plt.gca().yaxis.set_major_formatter(ticker.ScalarFormatter())
    plt.grid(which="minor")
    plt.title(f"Test RMSE (Dataset {data_No} / Qubit: {n_qubit} / Q_depth: {q_depth})", fontsize=16)

    fig.tight_layout()
    plt.xticks(fontsize=14)
    plt.yticks(fontsize=14)
    ax1.xaxis.label.set_fontsize(14)
    ax1.yaxis.label.set_fontsize(14)
    ax1.legend(fontsize=15)
    plt.savefig(f"../results/figure/test_RMSE_nq_{cc_key}_{q_depth}depth.png")
    plt.show()

In [ ]:
def plot_improvement(data_No, n_qubit, q_depth, ymax=None, ymin=None):
    df_improve = pd.DataFrame(columns=["RMSE (kcal/mol)", "n_layer", "Parameter"])

    base_key = f"s0{data_No}_{n_qubit}qu"

    for n_layer in range(1, 5):
        df_improve = pd.concat([df_improve, pd.DataFrame({"Improvement (kcal/mol)": calc_improve_value(cc_rmse, data_No, n_qubit, n_layer), "n_layer": n_layer, "Parameter": "Classic"})])
        df_improve = pd.concat([df_improve, pd.DataFrame({"Improvement (kcal/mol)": calc_improve_value(qc_rmse_Ry, data_No, n_qubit, n_layer, q_depth), "n_layer": n_layer, "Parameter": "Ry"})])
        df_improve = pd.concat([df_improve, pd.DataFrame({"Improvement (kcal/mol)": calc_improve_value(qc_rmse_RzRy, data_No, n_qubit, n_layer, q_depth), "n_layer": n_layer, "Parameter": "RzRy"})])
        df_improve = pd.concat([df_improve, pd.DataFrame({"Improvement (kcal/mol)": calc_improve_value(qc_rmse_Rot, data_No, n_qubit, n_layer, q_depth), "n_layer": n_layer, "Parameter": "RzRyRz"})])

    sns.set(style="darkgrid")
    sns.barplot(x="n_layer", y="Improvement (kcal/mol)", hue="Parameter", data=df_improve)

    if ymin is None:
        ymin = 0.01
    if ymax is None:
        ymax = max(df_improve["Improvement (kcal/mol)"].values) * 1.3
    plt.ylim(ymin, ymax)
    plt.xlabel("# of layer")
    plt.yscale("log")
    plt.yticks([0.1, 0.5, 1.0, 2.0])
    plt.gca().yaxis.set_minor_formatter(ticker.NullFormatter())
    plt.gca().yaxis.set_major_formatter(ticker.ScalarFormatter())
    plt.grid(which="minor")
    plt.title(f"Test Improvement (Data {data_No} / Qubit: {n_qubit} / Q_depth: {q_depth}")
    plt.tight_layout()
    plt.savefig(f"../results/figure/test_Improve_{base_key}_{q_depth}depth.png")
    plt.show()
    return df_improve

In [ ]:
for n_qubit in [4, 8, 12]:
    plot_rmse(1, n_qubit, 1, ymin=0.3, ymax=2.1, yrange=np.arange(2, 21, 2) * 0.1)

In [ ]:
for n_qubit in [4, 8, 12]:
    for q_depth in [1, 5]:
        plot_rmse(1, n_qubit, q_depth, ymax=2.1)

In [ ]:
def plot_rmse_nq(data_No, n_layer, q_depth, ymax=None, ymin=None):
    df_rmse = pd.DataFrame(columns=["RMSE (kcal/mol)", "n_qubit", "Parameter"])
    df_param = pd.DataFrame(columns=["# of parameter", "n_qubit", "Parameter"])

    base_key = f"s0{data_No}"

    for n_qubit in [4, 8, 12]:
        cc_key = base_key + f"_{n_qubit}qu_{n_layer}layer"
        qc_key = cc_key + f"_{q_depth}depth"

        for param_name, formula in param_formula.items():
            df_param = pd.concat([df_param, pd.DataFrame({"# of parameter": [formula(n_qubit)], "n_qubit": [n_qubit], "Parameter": [param_name]})])

        df_rmse = pd.concat([df_rmse, pd.DataFrame({"RMSE (kcal/mol)": cc_rmse[cc_key], "n_qubit": n_qubit, "Parameter": "Classical"})])
        df_rmse = pd.concat([df_rmse, pd.DataFrame({"RMSE (kcal/mol)": qc_rmse_Ry[qc_key], "n_qubit": n_qubit, "Parameter": "Ry"})])
        df_rmse = pd.concat([df_rmse, pd.DataFrame({"RMSE (kcal/mol)": qc_rmse_RzRy[qc_key], "n_qubit": n_qubit, "Parameter": "RzRy"})])
        #df_rmse = pd.concat([df_rmse, pd.DataFrame({"RMSE (kcal/mol)": qc_rmse_Rot[qc_key], "n_qubit": n_qubit, "Parameter": "RzRyRz"})])

    fig, ax1 = plt.subplots()

    sns.set(style="darkgrid")
    sns.barplot(x="n_qubit", y="RMSE (kcal/mol)", hue="Parameter", data=df_rmse, ax=ax1)
    ax1.axhline(1.0, color="k", ls="--")

    if ymin is None:
        ymin=0.3
    if ymax is None:
        ymax = max(df_rmse["RMSE (kcal/mol)"].values) * 1.3
    ax1.set_ylim(ymin, ymax)
    ax1.set_xlabel(r"$n_q$")
    #ax1.set_yscale("log")
    ax1.yaxis.set_minor_formatter(ticker.NullFormatter())
    ax1.yaxis.set_major_formatter(ticker.ScalarFormatter())
    ax1.grid(which="minor")
    ax1.set_title(f"Test RMSE (Dataset {data_No} / # of layer {n_layer} / Q_depth: {q_depth})", fontsize=16)

    """
    ax2 = ax1.twinx()
    for param_name, group in df_param.groupby("Parameter"):
        group = group.sort_values(by="n_qubit")
        for i, (_, row) in enumerate(group.iterrows()):
            ax2.plot(
                i + 0.2 * (ord(param_name[0]) - ord("A")),
                row["# of parameter"], color="k", marker="o"
                )
    ax2.set_yscale("log")
    """
    fig.tight_layout()
    plt.xticks(fontsize=14)
    plt.yticks(fontsize=14)
    ax1.xaxis.label.set_fontsize(14)
    ax1.yaxis.label.set_fontsize(14)
    ax1.legend(fontsize=15)
    plt.savefig(f"../results/figure/test_RMSE_nq_{cc_key}_{q_depth}depth.png")
    plt.show()

### s01

In [ ]:
for d in [1, 5]:
    plot_rmse_nq(1, 1, d, ymax=2.0, ymin=1.0)

In [ ]:
for l in [2, 3, 4]:
    for d in [1, 5]:
        plot_rmse_nq(1, l, d, ymax=0.8, ymin=0.4)

### s02

In [ ]:
for d in [1, 5]:
    plot_rmse_nq(2, 1, d, ymax=6.0, ymin=5.0)

In [ ]:
for l in [2, 3, 4]:
    for d in [1, 5]:
        plot_rmse_nq(2, l, d, ymax=1.3, ymin=0.7)

### s03

In [ ]:
for d in [1, 5]:
    plot_rmse_nq(3, 1, d, ymax=3.0, ymin=2.0)

In [ ]:
for l in [2, 3, 4]:
    for d in [1, 5]:
        plot_rmse_nq(3, l, d, ymax=1.0, ymin=0.5)

### s04

In [ ]:
for d in [1, 5]:
    plot_rmse_nq(4, 1, d, ymax=3.0, ymin=2.0)

In [ ]:
for l in [2, 3, 4]:
    for d in [1, 5]:
        plot_rmse_nq(4, l, d, ymax=1.0, ymin=0.5)

In [ ]:
for n_qubit in [4, 8, 12]:
    for q_depth in [1, 5]:
        df_improve = plot_improvement(1, n_qubit, q_depth, ymax=2.5)

In [ ]:
df_improve = plot_improvement(1, 8, 1, ymax=2.5)

In [ ]:
df_improve.query("(n_layer == 1) & (Parameter == 'RzRyRz')")["Improvement (kcal/mol)"].mean()

In [ ]:
df_improve.query("(n_layer == 2) & (Parameter == 'RzRyRz')")["Improvement (kcal/mol)"].mean()

In [ ]:
df_improve.query("(n_layer == 3) & (Parameter == 'RzRyRz')")["Improvement (kcal/mol)"].mean()

In [ ]:
df_improve.query("(n_layer == 4) & (Parameter == 'RzRyRz')")["Improvement (kcal/mol)"].mean()

## dベース

In [ ]:
def plot_rmse_d(data_No, n_qubit, ymax=None, ymin=None, yrange=None):
    df_rmse = pd.DataFrame(columns=["RMSE (kcal/mol)", "n_layer", "q_depth", "Parameter"])

    base_key = f"s0{data_No}_{n_qubit}qu"

    for n_layer in range(1, 5):
        for q_depth in [1, 5]:
            cc_key = base_key + f"_{n_layer}layer"
            qc_key = cc_key + f"_{q_depth}depth"

            df_rmse = pd.concat([df_rmse, pd.DataFrame({"RMSE (kcal/mol)": qc_rmse_RzRy[qc_key], "n_layer": n_layer, "q_depth": r"$d=$" + str(q_depth), "Parameter": "RzRy"})])

    fig, ax1 = plt.subplots()
    sns.set(style="darkgrid")
    sns.barplot(x="n_layer", y="RMSE (kcal/mol)", hue="q_depth", data=df_rmse)
    plt.axhline(1.0, color="k", ls="--")

    if ymin is None:
        ymin = 0.3
    if ymax is None:
        ymax = max(df_rmse["RMSE (kcal/mol)"].values) * 1.3
    plt.ylim(ymin, ymax)
    plt.xlabel("# of layer" + " " + r"$(l)$")
    #plt.yscale("log")
    #tick_list = list(np.arange(math.floor(ymax))).insert(1, 0.5)
    if yrange is not None:
        plt.yticks(yrange)
    plt.gca().yaxis.set_minor_formatter(ticker.NullFormatter())
    plt.gca().yaxis.set_major_formatter(ticker.ScalarFormatter())
    plt.grid(which="minor")
    plt.title(f"Test RMSE (Dataset {data_No} / Qubit: {n_qubit} / P: RzRy)", fontsize=16)

    fig.tight_layout()
    plt.xticks(fontsize=14)
    plt.yticks(fontsize=14)
    ax1.xaxis.label.set_fontsize(14)
    ax1.yaxis.label.set_fontsize(14)
    ax1.legend(fontsize=15)
    plt.savefig(f"../results/figure/test_RMSE_d_{base_key}.png")
    plt.show()

In [ ]:
plot_rmse_d(1, 4)

## datasetベース

In [ ]:
def plot_improvement_data(n_qubit, n_layer, q_depth, ymax=None, ymin=None):
    df_improve = pd.DataFrame(columns=["RMSE (kcal/mol)", "dataset", "Parameter"])

    for data_No in range(1, 5):
        df_improve = pd.concat([df_improve, pd.DataFrame({"Improvement (kcal/mol)": calc_improve_value(cc_rmse, data_No, n_qubit, n_layer), "dataset": data_No, "Parameter": "Classical"})])
        df_improve = pd.concat([df_improve, pd.DataFrame({"Improvement (kcal/mol)": calc_improve_value(qc_rmse_Ry, data_No, n_qubit, n_layer, q_depth), "dataset": data_No, "Parameter": "Ry"})])
        df_improve = pd.concat([df_improve, pd.DataFrame({"Improvement (kcal/mol)": calc_improve_value(qc_rmse_RzRy, data_No, n_qubit, n_layer, q_depth), "dataset": data_No, "Parameter": "RzRy"})])
        #df_improve = pd.concat([df_improve, pd.DataFrame({"Improvement (kcal/mol)": calc_improve_value(qc_rmse_Rot, data_No, n_qubit, n_layer, q_depth), "dataset": f"dataset {data_No}", "Parameter": "RzRyRz"})])

    fig, ax1 = plt.subplots()
    sns.set(style="darkgrid")
    sns.barplot(x="dataset", y="Improvement (kcal/mol)", hue="Parameter", data=df_improve)

    if ymin is None:
        ymin = 0.01
    if ymax is None:
        ymax = max(df_improve["Improvement (kcal/mol)"].values) * 1.3
    plt.ylim(ymin, ymax)
    plt.xlabel("dataset No.")
    #plt.yscale("log")
    plt.xticks(fontsize=14)
    plt.yticks(np.arange(0, 16.1, 5) * 0.1, fontsize=14)
    plt.gca().yaxis.set_minor_formatter(ticker.NullFormatter())
    plt.gca().yaxis.set_major_formatter(ticker.ScalarFormatter())
    plt.grid(which="minor")
    plt.title(f"Test Improvement (Qubit: {n_qubit} / # of layer: {n_layer} / Q_depth: {q_depth})", fontsize=16)
    ax1.xaxis.label.set_fontsize(14)
    ax1.yaxis.label.set_fontsize(14)
    ax1.legend(fontsize=15)
    plt.tight_layout()
    key = f"{n_qubit}qu_{n_layer}layer_{q_depth}depth"
    plt.savefig(f"../results/figure/test_Improve_data_{key}.png")
    plt.show()

In [ ]:
plot_improvement_data(4, 1, 1)

# データ数を増やした時

In [ ]:
with open("../results/test/CC_rmse_dict.pickle", "rb") as f:
    cc_rmse = pickle.load(f)
with open("../results/test/QC_rmse_dict_RzRy.pickle", "rb") as f:
    qc_rmse_5000 = pickle.load(f)
with open("../results/test/QC_rmse_dict_4_all_RzRy.pickle", "rb") as f:
    qc_rmse_all_4qu = pickle.load(f)
with open("../results/test/QC_rmse_dict_8_all_RzRy.pickle", "rb") as f:
    qc_rmse_all_8qu = pickle.load(f)
#with open("../results/test/QC_rmse_dict_12_all_Rot.pickle", "rb") as f:
#    qc_rmse_all_12qu = pickle.load(f)

In [ ]:
def calc_improve_value(rmse_dict, data_No, n_qubit, n_layer, q_depth=None):
    key = f"s0{data_No}_{n_qubit}qu_{n_layer}layer"
    if q_depth is not None:
        key += f"_{q_depth}depth"
    rmse_list = rmse_dict[key]
    improve_list = baseline_dict[f"{n_qubit}q_{n_layer}layer"] - np.array(rmse_list)
    return improve_list

In [ ]:
baseline_dict

In [ ]:
qc_rmse_5000

In [ ]:
df_rmse = pd.DataFrame(columns=["RMSE (kcal/mol)", "data_No", "n_layer", "n_qubit", "train_data"])
df_diff = pd.DataFrame(columns=["Diff_RMSE (kcal/mol)", "data_No", "n_layer", "n_qubit"])

q_depth = 1

# 4 qubit
n_qubit = 4
for data_No in range(2, 5):
    for n_layer in range(1, 4):
        base_key = f"s0{data_No}_{n_qubit}qu"
        qc_key = f"{base_key}_{n_layer}layer_{q_depth}depth"

        df_rmse = pd.concat([df_rmse, pd.DataFrame({"RMSE (kcal/mol)": qc_rmse_5000[qc_key], "data_No": data_No, "n_layer": n_layer, "n_qubit": n_qubit, "train_data": "extract"})])
        df_rmse = pd.concat([df_rmse, pd.DataFrame({"RMSE (kcal/mol)": qc_rmse_all_4qu[qc_key], "data_No": data_No, "n_layer": n_layer, "n_qubit": n_qubit, "train_data": "all"})])
        df_diff = pd.concat([df_diff, pd.DataFrame({"Diff_RMSE (kcal/mol)": np.array(qc_rmse_5000[qc_key]) - qc_rmse_all_4qu[qc_key][0], "data_No": data_No, "n_layer": n_layer, "n_qubit": n_qubit})])

n_qubit = 8
for data_No in range(2, 5):
    for n_layer in range(1, 4):
        base_key = f"s0{data_No}_{n_qubit}qu"
        qc_key = f"{base_key}_{n_layer}layer_{q_depth}depth"

        df_rmse = pd.concat([df_rmse, pd.DataFrame({"RMSE (kcal/mol)": qc_rmse_5000[qc_key], "data_No": data_No, "n_layer": n_layer, "n_qubit": n_qubit, "train_data": "extract"})])
        df_rmse = pd.concat([df_rmse, pd.DataFrame({"RMSE (kcal/mol)": qc_rmse_all_8qu[qc_key], "data_No": data_No, "n_layer": n_layer, "n_qubit": n_qubit, "train_data": "all"})])
        df_diff = pd.concat([df_diff, pd.DataFrame({"Diff_RMSE (kcal/mol)": np.array(qc_rmse_5000[qc_key]) - qc_rmse_all_8qu[qc_key][0], "data_No": data_No, "n_layer": n_layer, "n_qubit": n_qubit})])
"""
n_qubit = 12
for data_No in range(2, 4):
    for n_layer in range(1, 4):
        base_key = f"s0{data_No}_{n_qubit}qu"
        qc_key = f"{base_key}_{n_layer}layer_{q_depth}depth"

        df_rmse = pd.concat([df_rmse, pd.DataFrame({"RMSE (kcal/mol)": qc_rmse_5000[qc_key], "data_No": data_No, "n_layer": n_layer, "n_qubit": n_qubit, "train_data": "extract"})])
        df_rmse = pd.concat([df_rmse, pd.DataFrame({"RMSE (kcal/mol)": qc_rmse_all_12qu[qc_key], "data_No": data_No, "n_layer": n_layer, "n_qubit": n_qubit, "train_data": "all"})])
"""

In [ ]:
df_diff

In [ ]:
def plot_rmse(data_No, n_qubit, q_depth=1, ymax=None, ymin=None):
    fig, ax1 = plt.subplots()

    sns.set(style="darkgrid")
    sns.barplot(x="n_layer", y="RMSE (kcal/mol)", hue="train_data", data=df_rmse.query("(data_No == @data_No) & (n_qubit == @n_qubit)"))
    plt.axhline(1.0, color="k", ls="--")

    if ymin is None:
        ymin = 0.3
    if ymax is None:
        ymax = max(df_rmse["RMSE (kcal/mol)"].values) * 1.3
    plt.ylim(ymin, ymax)
    plt.xlabel("# of layer")
    plt.yscale("log")
    #tick_list = list(np.arange(math.floor(ymax))).insert(1, 0.5)
    plt.yticks([0.5, 1.0, 2.0])
    plt.gca().yaxis.set_minor_formatter(ticker.NullFormatter())
    plt.gca().yaxis.set_major_formatter(ticker.ScalarFormatter())
    plt.grid(which="minor")
    plt.title(f"Test RMSE (Dataset {data_No} / Qubit: {n_qubit} / Q_depth: {q_depth})", fontsize=16)

    plt.xticks(fontsize=14)
    plt.yticks(fontsize=14)
    ax1.xaxis.label.set_fontsize(14)
    ax1.yaxis.label.set_fontsize(14)
    ax1.legend(fontsize=15)
    plt.tight_layout()
    plt.savefig(f"../results/figure/test_RMSE_s0{data_No}_{n_qubit}qu_{q_depth}depth_extract_all.png")
    plt.show()

In [ ]:
def plot_diff_rmse(n_layer, n_qubit, q_depth=1, ymax=None, ymin=None):
    fig, ax1 = plt.subplots()

    sns.set(style="darkgrid")
    sns.barplot(x="data_No", y="Diff_RMSE (kcal/mol)", data=df_diff.query("(n_layer == @n_layer) & (n_qubit == @n_qubit)"))

    #if ymin is None:
    #    ymin = -0.3
    #if ymax is None:
    #    ymax = max(df_diff["Diff_RMSE (kcal/mol)"].values) * 1.3
    plt.ylim(ymin, ymax)
    plt.xlabel("dataset No.")
    #plt.yscale("log")
    #tick_list = list(np.arange(math.floor(ymax))).insert(1, 0.5)
    #plt.yticks([0.5, 1.0, 2.0])
    plt.gca().yaxis.set_minor_formatter(ticker.NullFormatter())
    plt.gca().yaxis.set_major_formatter(ticker.ScalarFormatter())
    plt.grid(which="minor")
    plt.title(f"Diff RMSE (# of layer: {n_layer} / Qubit: {n_qubit} / Q_depth: {q_depth})", fontsize=16)

    plt.xticks(fontsize=14)
    plt.yticks(fontsize=14)
    ax1.xaxis.label.set_fontsize(14)
    ax1.yaxis.label.set_fontsize(14)
    ax1.legend(fontsize=15)
    plt.tight_layout()
    plt.savefig(f"../results/figure/test_diff_RMSE_l{n_layer}_{n_qubit}qu_{q_depth}depth_extract_all.png")
    plt.show()

In [ ]:
for l in range(1, 4):
    plot_diff_rmse(l, 4)

In [ ]:
for n_qubit in [4, 8]:
    plot_rmse(2, n_qubit, ymax=6.1)

In [ ]:
for n_qubit in [4, 8]:
    plot_rmse(3, n_qubit, ymax=3.1)

In [ ]:
for n_qubit in [4, 8]:
    plot_rmse(4, n_qubit, ymax=3.1)

# パラメータ変化

## モデル定義

In [ ]:
%load_ext autoreload
%autoreload 2
from quantum_model import RotCircuit, RzRyCircuit, RyCircuit
from transfer_model import TransferANI

In [ ]:
# Constants, Construct AEV
Rcr = 5.2000e+00
Rca = 3.5000e+00
EtaR = torch.tensor([1.6000000e+01], device=device)
ShfR = torch.tensor([9.0000000e-01, 1.1687500e+00, 1.4375000e+00, 1.7062500e+00, 1.9750000e+00, 2.2437500e+00, 2.5125000e+00, 2.7812500e+00, 3.0500000e+00, 3.3187500e+00, 3.5875000e+00, 3.8562500e+00, 4.1250000e+00, 4.3937500e+00, 4.6625000e+00, 4.9312500e+00], device=device)
Zeta = torch.tensor([3.2000000e+01], device=device)
ShfZ = torch.tensor([1.9634954e-01, 5.8904862e-01, 9.8174770e-01, 1.3744468e+00, 1.7671459e+00, 2.1598449e+00, 2.5525440e+00, 2.9452431e+00], device=device)
EtaA = torch.tensor([8.0000000e+00], device=device)
ShfA = torch.tensor([9.0000000e-01, 1.5500000e+00, 2.2000000e+00, 2.8500000e+00], device=device)
species_order = ['H', 'C', 'N', 'O']
num_species = len(species_order)
aev_computer = torchani.AEVComputer(Rcr, Rca, EtaR, ShfR, EtaA, Zeta, ShfA, ShfZ, num_species)
energy_shifter = torchani.utils.EnergyShifter(None)
energy_shifter.self_energies = torch.Tensor([-0.6047, -38.0684, -54.7061, -75.1796])
energy_shifter = energy_shifter.to(device)

## 読み込み

In [ ]:
species = ["H", "C", "N", "O"]

In [ ]:
model_params = {
    "n_layer": np.arange(1, 5),
    "q_depth": (1, 5),
    "n_qubit": (4, 8, 12),
    "seed": np.arange(5),
}
keys, values = zip(*model_params.items())
params_product = [dict(zip(keys, p)) for p in product(*values)]

df_params = pd.DataFrame(columns=["params_change", "species", "n_layer", "n_qubit", "q_depth", "Parameter"])

In [ ]:
cnt = 0

In [ ]:
# Rot
for params in params_product:
    n_layer, q_depth, n_qubit, seed  = params.values()

    qc_dev = qml.device("default.qubit", wires=n_qubit)
    circuit = RotCircuit(qc_dev, q_depth)
    weight_shapes = {"weights": (q_depth, 2, n_qubit, 3)}

    ani_instance = TransferANI(n_layer, n_qubit, device, aev_computer)
    model, _, _ = ani_instance.transfer_qlayer(circuit, weight_shapes)
    quantum_path = f"../model/Quantum_transfer/best_QC_s01_{n_layer}layer_{n_qubit}qu_d{q_depth}_seed{seed}_Rot.pt"
    model.load_state_dict(torch.load(quantum_path, map_location=torch.device("cpu")))
    model.to(device)

    for s in species:
        param = model[1][s].state_dict()[f"{n_layer * 2}.weights"].cpu().data.numpy().reshape(q_depth * 2 * n_qubit, 3)
        df_params = pd.concat([df_params, pd.DataFrame({
            "params_change": np.linalg.norm(param, 2),
            # abs(model[1][s].state_dict()[f"{n_layer * 2}.weights"].cpu().data.numpy()),
            "species": s,
            "n_layer": n_layer,
            "n_qubit": n_qubit,
            "q_depth": q_depth,
            "Parameter": "RzRyRz",
        }, index=[cnt])])
        cnt += 1

In [ ]:
# RzRy
for params in params_product:
    n_layer, q_depth, n_qubit, seed  = params.values()

    qc_dev = qml.device("default.qubit", wires=n_qubit)
    circuit = RzRyCircuit(qc_dev, q_depth)
    weight_shapes = {"weights": (q_depth, 2, 2, n_qubit)}

    ani_instance = TransferANI(n_layer, n_qubit, device, aev_computer)
    model, _, _ = ani_instance.transfer_qlayer(circuit, weight_shapes)
    quantum_path = f"../model/Quantum_transfer/best_QC_s01_{n_layer}layer_{n_qubit}qu_d{q_depth}_seed{seed}_RzRy.pt"
    model.load_state_dict(torch.load(quantum_path, map_location=torch.device("cpu")))
    model.to(device)

    for s in species:
        param = model[1][s].state_dict()[f"{n_layer * 2}.weights"].cpu().data.numpy().reshape(q_depth * 2 * 2, n_qubit)
        split_weight = np.split(param, q_depth * 2, axis=0)
        split_T_weight = [w.T for w in split_weight]
        weight_set = np.concatenate(split_T_weight)

        df_params = pd.concat([df_params, pd.DataFrame({
            "params_change": np.linalg.norm(weight_set, 2),
            # abs(model[1][s].state_dict()[f"{n_layer * 2}.weights"].cpu().data.numpy()),
            "species": s,
            "n_layer": n_layer,
            "n_qubit": n_qubit,
            "q_depth": q_depth,
            "Parameter": "RzRy",
        }, index=[cnt])])
        cnt += 1

In [ ]:
# Ry
for params in params_product:
    n_layer, q_depth, n_qubit, seed  = params.values()

    qc_dev = qml.device("default.qubit", wires=n_qubit)
    circuit = RzRyCircuit(qc_dev, q_depth)
    weight_shapes = {"weights": (q_depth, 2, 1, n_qubit)}

    ani_instance = TransferANI(n_layer, n_qubit, device, aev_computer)
    model, _, _ = ani_instance.transfer_qlayer(circuit, weight_shapes)
    quantum_path = f"../model/Quantum_transfer/best_QC_s01_{n_layer}layer_{n_qubit}qu_d{q_depth}_seed{seed}_Ry.pt"
    model.load_state_dict(torch.load(quantum_path, map_location=torch.device("cpu")))
    model.to(device)

    for s in species:
        param = model[1][s].state_dict()[f"{n_layer * 2}.weights"].cpu().data.numpy().reshape(q_depth * 2 * 1, n_qubit)
        split_weight = np.split(param, q_depth * 2, axis=0)
        split_T_weight = [w.T for w in split_weight]
        weight_set = np.concatenate(split_T_weight)

        df_params = pd.concat([df_params, pd.DataFrame({
            "params_change": np.linalg.norm(weight_set, 2),
            # abs(model[1][s].state_dict()[f"{n_layer * 2}.weights"].cpu().data.numpy()),
            "species": s,
            "n_layer": n_layer,
            "n_qubit": n_qubit,
            "q_depth": q_depth,
            "Parameter": "Ry",
        }, index=[cnt])])
        cnt += 1

In [ ]:
df_params

## 描画

In [ ]:
sns.set(style="darkgrid")

In [ ]:
# colors = sns.color_palette()[1:]
n_qubit = 4

# 各元素
for s in species:
    fig = plt.figure()
    ax = fig.add_subplot(111)
    bp = sns.boxplot(
        x="Parameter", y="params_change", hue="n_layer",
        data=df_params.query("(species == @s) & (n_qubit == @n_qubit)"), ax=ax,
        #palette=colors
        )
    ax.set_title(f"Data: 1, Qubit: {n_qubit}, Atom: {s}")
    #ax.set_ylim(-0.01, 0.26)
    sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))
    plt.tight_layout()
    plt.savefig(f"../results/figure/params_change_s01_{n_qubit}qu_{s}.png")
    plt.show()

# 総和
fig = plt.figure()
ax = fig.add_subplot(111)
bp = sns.boxplot(
    x="Parameter", y="params_change", hue="n_layer",
    data=df_params.query("n_qubit == @n_qubit"), ax=ax)
ax.set_title(f"Data: 1, Qubit: {n_qubit}")
#ax.set_ylim(-0.01, 0.26)
sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))
plt.tight_layout()
plt.savefig(f"../results/figure/params_change_s01_{n_qubit}qu_all.png")
plt.show()

In [ ]:
# colors = sns.color_palette()[1:]
n_qubit = 8

# 各元素
for s in species:
    fig = plt.figure()
    ax = fig.add_subplot(111)
    bp = sns.boxplot(
        x="Parameter", y="params_change", hue="n_layer",
        data=df_params.query("(species == @s) & (n_qubit == @n_qubit)"), ax=ax,
        #palette=colors
        )
    ax.set_title(f"Data: 1, Qubit: {n_qubit}, Atom: {s}")
    #ax.set_ylim(-0.01, 0.26)
    sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))
    plt.tight_layout()
    plt.savefig(f"../results/figure/params_change_s01_{n_qubit}qu_{s}.png")
    plt.show()

# 総和
fig = plt.figure()
ax = fig.add_subplot(111)
bp = sns.boxplot(
    x="Parameter", y="params_change", hue="n_layer",
    data=df_params.query("n_qubit == @n_qubit"), ax=ax)
ax.set_title(f"Data: 1, Qubit: {n_qubit}")
#ax.set_ylim(-0.01, 0.26)
sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))
plt.tight_layout()
plt.savefig(f"../results/figure/params_change_s01_{n_qubit}qu_all.png")
plt.show()

In [ ]:
# colors = sns.color_palette()[1:]
n_qubit = 12

# 各元素
for s in species:
    fig = plt.figure()
    ax = fig.add_subplot(111)
    bp = sns.boxplot(
        x="Parameter", y="params_change", hue="n_layer",
        data=df_params.query("(species == @s) & (n_qubit == @n_qubit)"), ax=ax,
        #palette=colors
        )
    ax.set_title(f"Data: 1, Qubit: {n_qubit}, Atom: {s}")
    #ax.set_ylim(-0.01, 0.26)
    sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))
    plt.tight_layout()
    plt.savefig(f"../results/figure/params_change_s01_{n_qubit}qu_{s}.png")
    plt.show()

# 総和
fig = plt.figure()
ax = fig.add_subplot(111)
bp = sns.boxplot(
    x="Parameter", y="params_change", hue="n_layer",
    data=df_params.query("n_qubit == @n_qubit"), ax=ax)
ax.set_title(f"Data: 1, Qubit: {n_qubit}")
#ax.set_ylim(-0.01, 0.26)
sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))
plt.tight_layout()
plt.savefig(f"../results/figure/params_change_s01_{n_qubit}qu_all.png")
plt.show()

### depthを限定

In [ ]:
# colors = sns.color_palette()[1:]
n_qubit = 4

# 各元素
for s in species:
    fig = plt.figure()
    ax = fig.add_subplot(111)
    bp = sns.boxplot(
        x="Parameter", y="params_change", hue="n_layer",
        data=df_params.query("(species == @s) & (n_qubit == @n_qubit) & (q_depth == 1)"), ax=ax,
        #palette=colors
        )
    ax.set_title(f"Data: 1, Qubit: {n_qubit}, Depth: 1, Atom: {s}")
    #ax.set_ylim(-0.01, 0.26)
    sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))
    plt.tight_layout()
    plt.savefig(f"../results/figure/params_change_s01_{n_qubit}qu_1d_{s}.png")
    plt.show()

# 総和
fig = plt.figure()
ax = fig.add_subplot(111)
bp = sns.boxplot(
    x="Parameter", y="params_change", hue="n_layer",
    data=df_params.query("(n_qubit == @n_qubit) & (q_depth == 1)"), ax=ax)
ax.set_title(f"Data: 1, Qubit: {n_qubit}, Depth: 1")
#ax.set_ylim(-0.01, 0.26)
sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))
plt.tight_layout()
plt.savefig(f"../results/figure/params_change_s01_{n_qubit}qu_1d_all.png")
plt.show()

In [ ]:
# colors = sns.color_palette()[1:]
n_qubit = 8

# 各元素
for s in species:
    fig = plt.figure()
    ax = fig.add_subplot(111)
    bp = sns.boxplot(
        x="Parameter", y="params_change", hue="n_layer",
        data=df_params.query("(species == @s) & (n_qubit == @n_qubit) & (q_depth == 1)"), ax=ax,
        #palette=colors
        )
    ax.set_title(f"Data: 1, Qubit: {n_qubit}, Depth: 1, Atom: {s}")
    #ax.set_ylim(-0.01, 0.26)
    sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))
    plt.tight_layout()
    plt.savefig(f"../results/figure/params_change_s01_{n_qubit}qu_1d_{s}.png")
    plt.show()

# 総和
fig = plt.figure()
ax = fig.add_subplot(111)
bp = sns.boxplot(
    x="Parameter", y="params_change", hue="n_layer",
    data=df_params.query("(n_qubit == @n_qubit) & (q_depth == 1)"), ax=ax)
ax.set_title(f"Data: 1, Qubit: {n_qubit}, Depth: 1")
#ax.set_ylim(-0.01, 0.26)
sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))
plt.tight_layout()
plt.savefig(f"../results/figure/params_change_s01_{n_qubit}qu_1d_all.png")
plt.show()

In [ ]:
# colors = sns.color_palette()[1:]
n_qubit = 12

# 各元素
for s in species:
    fig = plt.figure()
    ax = fig.add_subplot(111)
    bp = sns.boxplot(
        x="Parameter", y="params_change", hue="n_layer",
        data=df_params.query("(species == @s) & (n_qubit == @n_qubit) & (q_depth == 1)"), ax=ax,
        #palette=colors
        )
    ax.set_title(f"Data: 1, Qubit: {n_qubit}, Depth: 1, Atom: {s}")
    #ax.set_ylim(-0.01, 0.26)
    sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))
    plt.tight_layout()
    plt.savefig(f"../results/figure/params_change_s01_{n_qubit}qu_1d_{s}.png")
    plt.show()

# 総和
fig = plt.figure()
ax = fig.add_subplot(111)
bp = sns.boxplot(
    x="Parameter", y="params_change", hue="n_layer",
    data=df_params.query("(n_qubit == @n_qubit) & (q_depth == 1)"), ax=ax)
ax.set_title(f"Data: 1, Qubit: {n_qubit}, Depth: 1")
#ax.set_ylim(-0.01, 0.26)
sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))
plt.tight_layout()
plt.savefig(f"../results/figure/params_change_s01_{n_qubit}qu_1d_all.png")
plt.show()

### depth=5

In [ ]:
# colors = sns.color_palette()[1:]
n_qubit = 4

# 各元素
for s in species:
    fig = plt.figure()
    ax = fig.add_subplot(111)
    bp = sns.boxplot(
        x="Parameter", y="params_change", hue="n_layer",
        data=df_params.query("(species == @s) & (n_qubit == @n_qubit) & (q_depth == 5)"), ax=ax,
        #palette=colors
        )
    ax.set_title(f"Data: 1, Qubit: {n_qubit}, Depth: 5, Atom: {s}")
    #ax.set_ylim(-0.01, 0.26)
    sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))
    plt.tight_layout()
    plt.savefig(f"../results/figure/params_change_s01_{n_qubit}qu_5d_{s}.png")
    plt.show()

# 総和
fig = plt.figure()
ax = fig.add_subplot(111)
bp = sns.boxplot(
    x="Parameter", y="params_change", hue="n_layer",
    data=df_params.query("(n_qubit == @n_qubit) & (q_depth == 5)"), ax=ax)
ax.set_title(f"Data: 1, Qubit: {n_qubit}, Depth: 5")
#ax.set_ylim(-0.01, 0.26)
sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))
plt.tight_layout()
plt.savefig(f"../results/figure/params_change_s01_{n_qubit}qu_5d_all.png")
plt.show()

In [ ]:
# colors = sns.color_palette()[1:]
n_qubit = 8

# 各元素
for s in species:
    fig = plt.figure()
    ax = fig.add_subplot(111)
    bp = sns.boxplot(
        x="Parameter", y="params_change", hue="n_layer",
        data=df_params.query("(species == @s) & (n_qubit == @n_qubit) & (q_depth == 5)"), ax=ax,
        #palette=colors
        )
    ax.set_title(f"Data: 1, Qubit: {n_qubit}, Depth: 1, Atom: {s}")
    #ax.set_ylim(-0.01, 0.26)
    sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))
    plt.tight_layout()
    plt.savefig(f"../results/figure/params_change_s01_{n_qubit}qu_5d_{s}.png")
    plt.show()

# 総和
fig = plt.figure()
ax = fig.add_subplot(111)
bp = sns.boxplot(
    x="Parameter", y="params_change", hue="n_layer",
    data=df_params.query("(n_qubit == @n_qubit) & (q_depth == 5)"), ax=ax)
ax.set_title(f"Data: 1, Qubit: {n_qubit}, Depth: 1")
#ax.set_ylim(-0.01, 0.26)
sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))
plt.tight_layout()
plt.savefig(f"../results/figure/params_change_s01_{n_qubit}qu_5d_all.png")
plt.show()

In [ ]:
# colors = sns.color_palette()[1:]
n_qubit = 12

# 各元素
for s in species:
    fig = plt.figure()
    ax = fig.add_subplot(111)
    bp = sns.boxplot(
        x="Parameter", y="params_change", hue="n_layer",
        data=df_params.query("(species == @s) & (n_qubit == @n_qubit) & (q_depth == 5)"), ax=ax,
        #palette=colors
        )
    ax.set_title(f"Data: 1, Qubit: {n_qubit}, Depth: 5, Atom: {s}")
    #ax.set_ylim(-0.01, 0.26)
    sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))
    plt.tight_layout()
    plt.savefig(f"../results/figure/params_change_s01_{n_qubit}qu_1d_{s}.png")
    plt.show()

# 総和
fig = plt.figure()
ax = fig.add_subplot(111)
bp = sns.boxplot(
    x="Parameter", y="params_change", hue="n_layer",
    data=df_params.query("(n_qubit == @n_qubit) & (q_depth == 5)"), ax=ax)
ax.set_title(f"Data: 1, Qubit: {n_qubit}, Depth: 5")
#ax.set_ylim(-0.01, 0.26)
sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))
plt.tight_layout()
plt.savefig(f"../results/figure/params_change_s01_{n_qubit}qu_5d_all.png")
plt.show()

In [ ]:
df_params.query("(n_qubit == @n_qubit) & (q_depth == 1) & (Parameter == 'RzRyRz')")

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(111)
bp = sns.boxplot(
    x="Parameter", y="params_change", hue="n_layer",
    data=df_params, ax=ax)
ax.set_title(f"Data: 1")
ax.set_ylim(-0.01, 0.26)
sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))
plt.tight_layout()
plt.savefig(f"../results/figure/params_change_s01_all_cnot.png")
plt.show()

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(111)
bp = sns.boxplot(
    x="Parameter", y="params_change", hue="n_layer",
    data=df_params, ax=ax)
ax.set_title(f"Data: 1")
ax.set_ylim(-0.01, 0.1)
sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))
plt.tight_layout()
# plt.savefig(f"../results/figure/params_change_s01_all_cnot.png")
plt.show()

# 原子間距離との関係調査

## データ

In [ ]:
def fix_seed(seed):
    # random
    random.seed(seed)
    # Numpy
    np.random.seed(seed)
    # Pytorch
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

In [ ]:
def subtract_self_energies_wo_ref(reenterable_iterable, self_energies, species_order):
    counts = {}
    Y = []
    for n, d in enumerate(reenterable_iterable):
        species = d["species"]
        count = Counter()
        for s in species:
            count[s] += 1
        for s, c in count.items():
            if s not in counts:
                counts[s] = [0] * n
            counts[s].append(c)
        for s in counts:
            if len(counts[s]) != n + 1:
                counts[s].append(0)
        Y.append(d["energies"])

    species = sorted(list(counts.keys()), key=lambda x: species_order.index(x))
    shifter = {s: e for s, e in zip(species, self_energies)}
    X = [counts[s] for s in species]

    def reenterable_iterable_factory():
        for d in reenterable_iterable:
            e = 0
            for s in d["species"]:
                e += shifter[s]
            d["energies"] -= e
            yield d
    return IterableAdapterWithLength(reenterable_iterable_factory, n)

In [ ]:
batch_size = 256

try:
    path = os.path.dirname(os.path.realpath(__file__))
except NameError:
    path = os.getcwd()

dspath = os.path.join(path, f"../data/ani_gdb_s01.h5")
data = torchani.data.load(dspath)
data_sub = TransformableIterable(subtract_self_energies_wo_ref(data, energy_shifter.self_energies, species_order))

fix_seed(42)
_, _, test = data_sub.species_to_indices(species_order).shuffle().split(0.6, 0.2, 0.2)
test_data = test.collate(batch_size).cache()

In [ ]:
species_s01_list = []
coordinates_s01_list = []

for t in test:
    sp = t["species"]
    coor = t["coordinates"]
    species_s01_list.append(sp)
    coordinates_s01_list.append(coor)

## モデル

In [ ]:
class Classicnet(nn.Module):
    def __init__(self, n_qubits):
        super().__init__()
        self.c_net = nn.Linear(n_qubits, n_qubits).to(device)
        self.act = torch.nn.CELU(0.1).to(device)
        #self.act = torch.nn.Tanh().to(device)

    def forward(self, input_features):
        #input_features = (input_features + 1) * 0.5
        c_out = self.c_net(input_features)
        c_out = self.act(c_out)
        #c_out = c_out * 2 - 1
        return c_out

In [ ]:
def change_final_layer2classic(model_origin, n_layer, n_qubits):

    model = copy.deepcopy(model_origin)

    final_number = (n_layer - 1) * 2

    # 古典層の追加: 量子との比較のため
    model[1].H[final_number] = Classicnet(n_qubits)
    model[1].C[final_number] = Classicnet(n_qubits)
    model[1].N[final_number] = Classicnet(n_qubits)
    model[1].O[final_number] = Classicnet(n_qubits)

    # 元々の最終層を復元
    model[1].H.add_module(str(final_number + 1), nn.Linear(n_qubits, 1))
    model[1].C.add_module(str(final_number + 1), nn.Linear(n_qubits, 1))
    model[1].N.add_module(str(final_number + 1), nn.Linear(n_qubits, 1))
    model[1].O.add_module(str(final_number + 1), nn.Linear(n_qubits, 1))

    return model

In [ ]:
mse = torch.nn.MSELoss(reduction='none')
mse_sum = torch.nn.MSELoss(reduction='sum')

def validate(model, validation):
    total_mse = 0.0
    count = 0
    model.train(False)

    true_array = np.array([])
    pred_array = np.array([])
    with torch.no_grad():
        for properties in validation:
            species = properties['species'].to(device)
            coordinates = properties['coordinates'].to(device).float()
            true_energies = properties['energies'].to(device).float()
            _, predicted_energies = model((species, coordinates))

            true_array = np.append(true_array, true_energies.cpu().data.numpy())
            pred_array = np.append(pred_array, predicted_energies.cpu().data.numpy())

            total_mse += mse_sum(predicted_energies, true_energies).item()
            count += predicted_energies.shape[0]

    rmse_kcalmol = hartree2kcalmol(math.sqrt(total_mse / count))
    model.train(True)

    return rmse_kcalmol, true_array, pred_array

## $l=1$

### モデル読み込み

In [ ]:
n_layer, q_depth, seed  = (1, 5, 0)

ani_nn = AniModel(n_layer + 1, n_qubits=4)
model_quantum = nn.Sequential(aev_computer, ani_nn).to(device)
model_quantum = change_final_layer2quantum(model_quantum, Quantumnet_rot, n_layer + 1, 4, q_depth)
quantum_path = f"../model/Quantum_transfer/best_QC_s01_{n_layer + 1}layer_d{q_depth}_seed{seed}_Rot_cnot.pt"
model_quantum.load_state_dict(torch.load(quantum_path, map_location=torch.device("cpu")))
model_quantum.to(device)

In [ ]:
n_layer, q_depth, seed  = (1, 5, 0)

ani_nn = AniModel(n_layer + 1, n_qubits=4)
model_classic = nn.Sequential(aev_computer, ani_nn).to(device)
model_classic = change_final_layer2classic(model_classic, n_layer + 1, 4)
#classic_path = f"../model/Classic_transfer/best_Classic_transfer_s01_{n_layer + 1}layer_seed{seed}_lr0.0001_5000sample_un_selfE_shuffle.pt"
classic_path = f"../model/Classic_transfer/best_Classic_transfer_s01_{n_layer + 1}layer_seed{seed}_lr0.0001.pt"
model_classic.load_state_dict(torch.load(classic_path, map_location=torch.device("cpu")))
model_classic.to(device)

### 精度検証

In [ ]:
def yy_plot(y_test, y_pred, ymax=None, ymin=None, name=None, title=None, fontsize=14):
    fig = plt.figure(figsize=(6, 6))
    ax = fig.add_subplot(1, 1, 1)
    plt.rcParams['font.family'] = 'Arial'
    plt.rcParams['mathtext.fontset'] = 'cm'
    plt.rcParams['axes.linewidth'] = 1.0

    plt.scatter(y_test, y_pred, alpha=0.3, marker='.')
    if (ymax is not None) and (ymin is not None):
        plt.plot([ymin, ymax], [ymin, ymax], 'k-')
        plt.ylim(ymin, ymax)
        plt.xlim(ymin, ymax)
    else:
        ymax = max(y_test.max(), y_pred.max())
        ymin = min(y_test.min(), y_pred.min())
        plt.plot([ymin - 0.05 * (ymax - ymin), ymax + 0.05 * (ymax - ymin)],
                 [ymin - 0.05 * (ymax - ymin), ymax + 0.05 * (ymax - ymin)], 'k-')
        plt.ylim(ymin - 0.05 * (ymax - ymin), ymax + 0.05 * (ymax - ymin))
        plt.xlim(ymin - 0.05 * (ymax - ymin), ymax + 0.05 * (ymax - ymin))
    plt.grid(linewidth=0.5)
    plt.tick_params(labelsize=fontsize)
    if title is not None:
        plt.title(title, fontsize=fontsize + 2)
    plt.xlabel('True', fontsize=fontsize)
    plt.ylabel('Pred', fontsize=fontsize)
    #plt.legend(fontsize=fontsize - 2)
    ax.set_aspect('equal')
    plt.tight_layout()
    if name is not None:
        plt.savefig(name, dpi=100)
    plt.show()

In [ ]:
qc_rmse, y_true, qc_pred = validate(model_quantum, test_data)

In [ ]:
cc_rmse, _, cc_pred = validate(model_classic, test_data)

In [ ]:
qc_rmse

In [ ]:
np.save("../results/s01_qc_pred_l1.npy", qc_rmse)

In [ ]:
cc_rmse

In [ ]:
yy_plot(y_true, qc_pred, title="s1_QC")

In [ ]:
yy_plot(y_true, cc_pred, title="s1_CC")

In [ ]:
qc_error_idx_1 = np.where(abs(y_true - qc_pred) > 0.01)[0]
len(qc_error_idx_1)

In [ ]:
cc_error_idx_1 = np.where(abs(y_true - cc_pred) > 0.01)[0]
len(cc_error_idx_1)

In [ ]:
qc_error_idx_1

In [ ]:
for i, properties in enumerate(test):
    if i in qc_error_idx_1:
        print(properties["species"])

In [ ]:
from ase import Atoms

In [ ]:
def transform_atom(l):
    transform_list = [1, 6, 7, 8, 0] # H, C, N, O, -1
    atom_num_list = [transform_list[i] for i in l]
    return atom_num_list

In [ ]:
qc_error_atom_dist = {'CH4': [], 'NH3': [], 'H2O': []}
qc_error_atom_angle = {'CH4': [], 'NH3': [], 'H2O': []}

### H2O

In [ ]:
error_H2O_dists = {"O-H": []}
qc_error_H2O_values = []
cc_error_H2O_values = []
H2O_coordinates = []

In [ ]:
for i in range(len(y_true)):
    if len(species_s01_list[i]) == 3:
        error_atoms = Atoms("N3", [(0, 0, 0), (1, 0, 0), (0, 0, 1)])

        error_atoms.set_positions(coordinates_s01_list[i])
        error_atoms.set_atomic_numbers(transform_atom(species_s01_list[i]))

        error_H2O_dists["O-H"].append(error_atoms.get_distance(0, 1))
        error_H2O_dists["O-H"].append(error_atoms.get_distance(0, 2))

        H2O_coordinates.append(coordinates_s01_list[i])
        qc_error_H2O_values.append(hartree2kcalmol(abs(qc_pred[i] - y_true[i])))
        cc_error_H2O_values.append(hartree2kcalmol(abs(cc_pred[i] - y_true[i])))

In [ ]:
len(qc_error_H2O_values)

In [ ]:
sum(np.array(qc_error_H2O_values) <= 1)

In [ ]:
sum(np.array(cc_error_H2O_values) <= 1)

#### 図示

In [ ]:
error_H2O_dists["maxO-H"] = []
it = iter(error_H2O_dists["O-H"])
for i, j in zip(it, it):
    error_H2O_dists["maxO-H"].append(max(i, j))

In [ ]:
error_H2O_dists["minO-H"] = []
it = iter(error_H2O_dists["O-H"])
for i, j in zip(it, it):
    error_H2O_dists["minO-H"].append(min(i, j))

In [ ]:
error_H2O_dists["diffO-H"] = []
it = iter(error_H2O_dists["O-H"])
for i, j in zip(it, it):
    error_H2O_dists["diffO-H"].append(abs(i - j))

In [ ]:
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(1, 1, 1)
plt.scatter(error_H2O_dists["maxO-H"], qc_error_H2O_values, alpha=0.7, marker='.')

plt.tick_params(labelsize=14)
plt.xlabel('H2O max O-H distance', fontsize=14)
plt.ylabel('|Pred - True| (kcal/mol)', fontsize=14)
plt.ylim(-0.5, 17)
plt.title("Max O-H vs Error (QC, $l=1$)")
plt.tight_layout()
#plt.savefig('../results/picture/H2O_maxO-Hdist_vs_error.png', dpi=100)
plt.show()

In [ ]:
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(1, 1, 1)
plt.scatter(error_H2O_dists["maxO-H"], cc_error_H2O_values, alpha=0.7, marker='.')

plt.tick_params(labelsize=14)
plt.xlabel('H2O max O-H distance', fontsize=14)
plt.ylabel('|Pred - True| (kcal/mol)', fontsize=14)
plt.ylim(-0.5, 17)
plt.title("Max O-H vs Error (CC, $l=1$)")
plt.tight_layout()
#plt.savefig('../results/picture/H2O_maxO-Hdist_vs_error.png', dpi=100)
plt.show()

In [ ]:
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(1, 1, 1)
plt.scatter(error_H2O_dists["minO-H"], qc_error_H2O_values, alpha=0.7, marker='.')

plt.tick_params(labelsize=14)
plt.xlabel('H2O min O-H distance', fontsize=14)
plt.ylabel('|Pred - True| (kcal/mol)', fontsize=14)
plt.ylim(-0.5, 17)
plt.title("min O-H vs Error (QC, $l=1$)")
plt.tight_layout()
#plt.savefig('../results/picture/H2O_minO-Hdist_vs_error.png', dpi=100)
plt.show()

In [ ]:
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(1, 1, 1)
plt.scatter(error_H2O_dists["minO-H"], cc_error_H2O_values, alpha=0.7, marker='.')

plt.tick_params(labelsize=14)
plt.xlabel('H2O min O-H distance', fontsize=14)
plt.ylabel('|Pred - True| (kcal/mol)', fontsize=14)
plt.ylim(-0.5, 17)
plt.title("min O-H vs Error (CC, $l=1$)")
plt.tight_layout()
#plt.savefig('../results/picture/H2O_minO-Hdist_vs_error.png', dpi=100)
plt.show()

In [ ]:
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(1, 1, 1)
plt.scatter(error_H2O_dists["diffO-H"], qc_error_H2O_values, alpha=0.7, marker='.')

plt.tick_params(labelsize=14)
plt.xlabel('H2O diff O-H distance', fontsize=14)
plt.ylabel('|Pred - True| (kcal/mol)', fontsize=14)
plt.ylim(-0.5, 17)
plt.title("Diff O-H vs Error (QC, $l=1$)")
plt.tight_layout()
#plt.savefig('../results/picture/H2O_diffO-Hdist_vs_error.png', dpi=100)
plt.show()

In [ ]:
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(1, 1, 1)
plt.scatter(error_H2O_dists["diffO-H"], cc_error_H2O_values, alpha=0.7, marker='.')

plt.tick_params(labelsize=14)
plt.xlabel('H2O diff O-H distance', fontsize=14)
plt.ylabel('|Pred - True| (kcal/mol)', fontsize=14)
plt.ylim(-0.5, 17)
plt.title("Diff O-H vs Error (CC, $l=1$)")
plt.tight_layout()
#plt.savefig('../results/picture/H2O_diffO-Hdist_vs_error.png', dpi=100)
plt.show()

## $l=2$

### モデル読み込み

In [ ]:
n_layer, q_depth, seed  = (2, 5, 0)

ani_nn = AniModel(n_layer + 1, n_qubits=4)
model_quantum_2 = nn.Sequential(aev_computer, ani_nn).to(device)
model_quantum_2 = change_final_layer2quantum(model_quantum_2, Quantumnet_rot, n_layer + 1, 4, q_depth)
quantum_path_2 = f"../model/Quantum_transfer/best_QC_s01_{n_layer + 1}layer_d{q_depth}_seed{seed}_Rot_cnot.pt"
model_quantum_2.load_state_dict(torch.load(quantum_path_2, map_location=torch.device("cpu")))
model_quantum_2.to(device)

In [ ]:
ani_nn = AniModel(n_layer + 1, n_qubits=4)
model_classic_2 = nn.Sequential(aev_computer, ani_nn).to(device)
model_classic_2 = change_final_layer2classic(model_classic_2, n_layer + 1, 4)
classic_path_2 = f"../model/Classic_transfer/best_Classic_transfer_s01_{n_layer + 1}layer_seed1_lr0.0001_5000sample_un_selfE_shuffle.pt"
model_classic_2.load_state_dict(torch.load(classic_path_2, map_location=torch.device("cpu")))
model_classic_2.to(device)

### 精度検証

In [ ]:
qc_rmse_2, _, qc_pred_2 = validate(model_quantum_2, test_data)

In [ ]:
cc_rmse_2, _, cc_pred_2 = validate(model_classic_2, test_data)

In [ ]:
np.save("../results/s01_qc_pred_l2.npy", qc_rmse_2)

In [ ]:
qc_rmse_2

In [ ]:
cc_rmse_2

In [ ]:
yy_plot(y_true, qc_pred_2, title="s1_QC")

In [ ]:
yy_plot(y_true, cc_pred_2, title="s1_CC")

In [ ]:
sum(hartree2kcalmol(abs(y_true - qc_pred_2)) < 1.0)

In [ ]:
sum(hartree2kcalmol(abs(y_true - cc_pred_2)) < 1.0)

### H2O

In [ ]:
qc_error_H2O_values_2 = []
cc_error_H2O_values_2 = []

In [ ]:
for i in range(len(y_true)):
    if len(species_s01_list[i]) == 3:
        qc_error_H2O_values_2.append(hartree2kcalmol(abs(qc_pred_2[i] - y_true[i])))
        cc_error_H2O_values_2.append(hartree2kcalmol(abs(cc_pred_2[i] - y_true[i])))

In [ ]:
sum(np.array(qc_error_H2O_values_2) <= 1)

In [ ]:
sum(np.array(cc_error_H2O_values_2) <= 1)

#### 図示

In [ ]:
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(1, 1, 1)
plt.scatter(error_H2O_dists["maxO-H"], qc_error_H2O_values_2, alpha=0.7, marker='.')

plt.tick_params(labelsize=14)
plt.xlabel('H2O max O-H distance', fontsize=14)
plt.ylabel('|Pred - True| (kcal/mol)', fontsize=14)
plt.ylim(-0.5, 3.5)
plt.title("Max O-H vs Error (QC, $l=1$)")
plt.tight_layout()
#plt.savefig('../results/picture/H2O_maxO-Hdist_vs_error.png', dpi=100)
plt.show()

In [ ]:
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(1, 1, 1)
plt.scatter(error_H2O_dists["maxO-H"], cc_error_H2O_values_2, alpha=0.7, marker='.')

plt.tick_params(labelsize=14)
plt.xlabel('H2O max O-H distance', fontsize=14)
plt.ylabel('|Pred - True| (kcal/mol)', fontsize=14)
plt.ylim(-0.5, 3.5)
plt.title("Max O-H vs Error (CC, $l=1$)")
plt.tight_layout()
#plt.savefig('../results/picture/H2O_maxO-Hdist_vs_error.png', dpi=100)
plt.show()

In [ ]:
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(1, 1, 1)
plt.scatter(error_H2O_dists["minO-H"], qc_error_H2O_values_2, alpha=0.7, marker='.')

plt.tick_params(labelsize=14)
plt.xlabel('H2O min O-H distance', fontsize=14)
plt.ylabel('|Pred - True| (kcal/mol)', fontsize=14)
plt.ylim(-0.5, 3.5)
plt.title("min O-H vs Error (QC, $l=1$)")
plt.tight_layout()
#plt.savefig('../results/picture/H2O_minO-Hdist_vs_error.png', dpi=100)
plt.show()

In [ ]:
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(1, 1, 1)
plt.scatter(error_H2O_dists["minO-H"], cc_error_H2O_values_2, alpha=0.7, marker='.')

plt.tick_params(labelsize=14)
plt.xlabel('H2O min O-H distance', fontsize=14)
plt.ylabel('|Pred - True| (kcal/mol)', fontsize=14)
plt.ylim(-0.5, 3.5)
plt.title("min O-H vs Error (CC, $l=1$)")
plt.tight_layout()
#plt.savefig('../results/picture/H2O_minO-Hdist_vs_error.png', dpi=100)
plt.show()

In [ ]:
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(1, 1, 1)
plt.scatter(error_H2O_dists["diffO-H"], qc_error_H2O_values_2, alpha=0.7, marker='.')

plt.tick_params(labelsize=14)
plt.xlabel('H2O diff O-H distance', fontsize=14)
plt.ylabel('|Pred - True| (kcal/mol)', fontsize=14)
plt.ylim(-0.5, 3.5)
plt.title("Diff O-H vs Error (QC, $l=1$)")
plt.tight_layout()
#plt.savefig('../results/picture/H2O_diffO-Hdist_vs_error.png', dpi=100)
plt.show()

In [ ]:
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(1, 1, 1)
plt.scatter(error_H2O_dists["diffO-H"], cc_error_H2O_values_2, alpha=0.7, marker='.')

plt.tick_params(labelsize=14)
plt.xlabel('H2O diff O-H distance', fontsize=14)
plt.ylabel('|Pred - True| (kcal/mol)', fontsize=14)
plt.ylim(-0.5, 3.5)
plt.title("Diff O-H vs Error (CC, $l=1$)")
plt.tight_layout()
#plt.savefig('../results/picture/H2O_diffO-Hdist_vs_error.png', dpi=100)
plt.show()

In [ ]:
model_aev = nn.Sequential(aev_computer).to(device)

In [ ]:
aev_params = np.array([])
for properties in test_data:
    species = properties["species"].to(device)
    coordinates = properties["coordinates"].to(device).float()
    aev = model_aev((species, coordinates))
    aev_params = np.append(aev_params, aev)

In [ ]:
aev_params[1].shape

In [ ]:
species